# Construction attributes and exposure tolerance — OpenView 3D




In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
PACKAGE_ROOT = next(
    path for path in [cwd, *cwd.parents]
    if (path / 'environment.yml').exists() and (path / 'src').exists()
)
sys.path.insert(0, str(PACKAGE_ROOT))

import pandas as pd
from IPython.display import Markdown, display

from src.figure_paths import figure_dir, save_figure
from src.analysis.construction import (
    attribute_unknownness_by_outcome, cross_fire_standardization,
    fit_damage_stages, fit_overall_attributes, prepare_construction_samples,
    publication_table_markdown,
)
from src.paths import DATA_DIR, DERIVED_3D

DATA = DATA_DIR
DERIVED = DERIVED_3D
ANALYSIS_PATH = DERIVED / 'analysis_openview3d.parquet'
BUILDING_CACHE = DERIVED / 'county_buildings.parquet'
RESULTS = DERIVED / 'construction'
SUPPLEMENTARY = figure_dir('supplementary')
RESULTS.mkdir(parents=True, exist_ok=True)
N_BOOT = int(os.environ.get('OPENVIEW_CONSTRUCTION_BOOTSTRAPS', '200'))
BOOTSTRAP_SEED = 20250729


## Population and attribute scaling

The analysis is restricted to structures with positive OpenView 3D realized coupling, $F^*>0$, and excludes documented defense from the principal construction models. Roof and vegetation estimates use structures with known roof material. Building-vintage and stage models additionally require construction year. Vegetation is scaled to one interquartile-range increase and construction year to one decade.

In [ ]:
if not ANALYSIS_PATH.exists():
    raise FileNotFoundError(
        f'OpenView 3D analysis table is missing: {ANALYSIS_PATH}. '
        'Run 00_population_sample.ipynb first.'
    )
analysis = pd.read_parquet(ANALYSIS_PATH)
required_analysis = {
    'graph_id', 'fire', 'exposed', 'F_destroyed_wmean', 'is_destroyed',
    'any_damage', 'outcome', 'roof_class', 'defended', 'ndvi_mean',
    'year_built', 'grid_id', 'graph_neighbors',
}
missing_analysis = required_analysis.difference(analysis.columns)
if missing_analysis:
    raise ValueError(
        f'OpenView 3D analysis is missing fields: {sorted(missing_analysis)}'
    )
if not analysis.graph_id.is_unique:
    raise ValueError('OpenView 3D analysis must contain one row per graph_id.')

samples = prepare_construction_samples(analysis)
sample_flow = pd.DataFrame([
    {'population': 'Assessed structures', 'n': len(analysis)},
    {'population': 'Exposed structures (F* > 0)', 'n': len(samples.exposed)},
    {'population': 'Exposed and undefended', 'n': len(samples.undefended)},
    {'population': 'Undefended with known roof', 'n': len(samples.known_roof)},
    {'population': 'Complete roof, vegetation and vintage', 'n': len(samples.complete)},
    {'population': 'Complete and damaged (escalation)',
     'n': int(samples.complete.any_damage.sum())},
])
sample_flow.to_csv(RESULTS / 'construction_sample_flow.csv', index=False)
display(sample_flow)
print(f'Analysis input: {ANALYSIS_PATH}')
print(f'NDVI IQR = {samples.ndvi_iqr:.4f}')
print(f'Construction year centered at {samples.year_reference:.0f}; reported per decade')

## Post-fire attribute-recording audit

*Supplement — Supplementary Table S1.*

Eaves, vent screening and window panes can become unobservable when a structure is destroyed. Unknown-recording rates are therefore compared between destroyed structures and structures that survived, including undamaged and partially damaged buildings. This audit uses all assessed structures in the updated OpenView analysis table because it evaluates DINS data availability rather than an exposure-response estimand. Strong outcome-dependent missingness precludes interpreting these fields as pre-fire construction attributes. Exterior siding is included for comparison but is also excluded from inference because the material remaining after fire may not represent the pre-fire assembly.

In [ ]:
attribute_recording = attribute_unknownness_by_outcome(analysis)
attribute_recording.to_csv(
    RESULTS / 'tableS_attribute_recording_by_outcome.csv', index=False
)
display(attribute_recording[[
    'attribute', 'survived_n', 'survived_unknown_n',
    'survived_unknown_percent', 'destroyed_n', 'destroyed_unknown_n',
    'destroyed_unknown_percent', 'difference_percentage_points',
]].round(1))

## Table 1a — Overall destruction

*Superseded — retained for comparison with the published analysis.*

Logistic models adjust for log OpenView 3D $F^*$ and fire. Roof and vegetation are mutually adjusted. The vintage model additionally includes continuous construction year. Attribute coefficients are converted to exposure-tolerance ratios using $\exp(-\gamma/\beta)$, where $\gamma$ is the attribute coefficient and $\beta$ is the log-exposure coefficient.

In [ ]:
overall_table, overall_raw, overall_models = fit_overall_attributes(samples)
overall_table.to_csv(RESULTS / 'table1a_construction_overall.csv', index=False)
display(overall_table[[
    'attribute', 'contrast', 'analysis_n', 'tolerance_ratio',
    'ci_lo', 'ci_hi', 'p_value',
]].round(4))

## Table 1b — Damage onset and escalation

*Superseded — retained for comparison with the published analysis.*

Onset is any recorded damage among exposed structures. Escalation is destruction conditional on having sustained damage. Both models contain roof, vegetation, continuous building vintage, log exposure and fire. A stacked model tests whether each association differs between stages. Escalation estimates condition on an observed outcome and are interpreted descriptively.

In [ ]:
stage_table, stage_raw, stage_models = fit_damage_stages(samples)
stage_table.to_csv(RESULTS / 'table1b_construction_stages.csv', index=False)
display(stage_table.round(4))

## Fitted models

*Neither — fits and diagnostics feeding every table and figure below.*

Five specifications are fitted. The first three use a common complete-data
sample of exposed, undefended structures and model destruction, any damage,
and destruction conditional on damage. They include roof material, continuous
$\log(\mathrm{NDVI})$, construction vintage, log realized coupling, OpenView
graph connectivity, distance to the cleaned fire-clipped CAL FIRE Influence
boundary and fire. The fourth model adds interactions between log coupling and
the property and site characteristics. The fifth restores defended structures
and estimates an additive documented-defense association.

The boundary distance follows the original urban-depth construction: a 100 m
morphological opening of the perimeter-clipped Influence Zone followed by the
shortest distance from each building's representative point. It is computed
directly from the OpenView building geometries so the notebook remains
self-contained. Continuous predictors
are centered at their sample means, which leaves every slope unchanged but
evaluates the interaction-model main effects at mean exposure rather than at
$F^*=1$, a value outside the data. Each added block is tested jointly with a
Wald test on the cluster-robust covariance rather than a likelihood-ratio
test, which clustering would invalidate. Every term is exported with its
coefficient, cluster-robust standard error, $z$ statistic, $P$ value, 95%
confidence interval and odds ratio, and the statsmodels report is printed for
each fit.

In [ ]:
import geopandas as gpd
import numpy as np
import shapely

from src.analysis.construction import ROOF_TERM, TILE_TERM, fit_clustered_logit

M_TO_FT = 3.28084
INFLUENCE_OPENING_M = 100.0  # shrink then expand to remove narrow slivers

# Distance to the cleaned, fire-clipped CAL FIRE Influence boundary. Building
# geometry is keyed by OpenView graph_id so the site covariate and F* come from
# the same graph population.
wui_zones = gpd.read_file(DATA / 'calfire_wui_la.gpkg').to_crs(26911)
influence_zone = wui_zones[wui_zones.WUI_DESC.eq('Influence Zone')]
fire_perimeters = (gpd.read_parquet(DATA / 'nx' / 'fire_perims.parquet')
                   .to_crs(26911).set_index('FIRE_NAME'))
county_buildings = (
    gpd.read_parquet(BUILDING_CACHE, columns=['building_id', 'geometry'])
    .rename(columns={'building_id': 'graph_id'})
    .to_crs(26911)
)
assessed_ids = set(analysis.graph_id.astype('int64'))
county_buildings = county_buildings[
    county_buildings.graph_id.isin(assessed_ids)
].drop_duplicates('graph_id').reset_index(drop=True)

depth_parts = []
for fire_name in ['EATON', 'PALISADES']:
    fire_ids = set(
        analysis.loc[analysis.fire.eq(fire_name), 'graph_id'].astype('int64')
    )
    buildings = county_buildings[
        county_buildings.graph_id.isin(fire_ids)
    ].copy()
    clipped = shapely.intersection(
        influence_zone.geometry.union_all(),
        fire_perimeters.loc[fire_name].geometry,
    )
    opened = shapely.buffer(
        shapely.buffer(clipped, -INFLUENCE_OPENING_M),
        INFLUENCE_OPENING_M,
    )
    boundary_parts = (
        gpd.GeoSeries([opened], crs=26911).boundary
        .explode(index_parts=False).reset_index(drop=True)
    )
    points = buildings.geometry.representative_point().to_numpy()
    nearest = boundary_parts.sindex.nearest(
        gpd.GeoSeries(points, crs=26911), return_all=False,
    )
    nearest_part = np.empty(len(buildings), dtype=int)
    nearest_part[nearest[0]] = nearest[1]
    depth_parts.append(pd.DataFrame({
        'graph_id': buildings.graph_id.astype('int64'),
        'fire': fire_name,
        'influence_depth_ft': shapely.length(shapely.shortest_line(
            boundary_parts.to_numpy()[nearest_part], points)) * M_TO_FT,
    }))
influence_depth = pd.concat(depth_parts, ignore_index=True)

# Structure size. Footprint area is the size measure carried here: facade and
# total surface area are the denominator of F* itself, so conditioning on them
# would partly undo the exposure normalization. Volume is kept alongside as an
# alternative; facade area is available in the run's building_metrics.
building_size = county_buildings[['graph_id']].copy()
building_size['footprint_m2'] = county_buildings.geometry.area
building_size = building_size.merge(
    pd.read_parquet(BUILDING_CACHE, columns=['building_id', 'height_m'])
    .rename(columns={'building_id': 'graph_id'}),
    on='graph_id', how='left', validate='one_to_one',
)
building_size['volume_m3'] = building_size.footprint_m2 * building_size.height_m

# graph_neighbors is the number of directional receiver-neighbor links in the
# updated OpenView building graph, independent of the neighbor's fire outcome.
def _prepare_sample(frame: pd.DataFrame) -> pd.DataFrame:
    merged = frame.copy()
    merged['graph_id'] = merged.graph_id.astype('int64')
    merged = merged.merge(
        influence_depth, on=['graph_id', 'fire'], how='left',
        validate='one_to_one',
    )
    merged = merged.merge(
        building_size[['graph_id', 'footprint_m2', 'volume_m3']],
        on='graph_id', how='left', validate='one_to_one',
    )
    merged = merged[
        merged.influence_depth_ft.gt(0) & merged.ndvi_mean.gt(0)
        & merged.graph_neighbors.gt(0) & merged.footprint_m2.gt(0)
    ].copy()
    merged['log_ndvi'] = np.log(merged.ndvi_mean)
    merged['log_influence_depth'] = np.log(merged.influence_depth_ft)
    merged['defended_structure'] = merged.defended.astype(int)
    merged['n_visible'] = merged.graph_neighbors.astype(float)
    merged['log_n_visible'] = np.log(merged.n_visible)
    merged['log_footprint'] = np.log(merged.footprint_m2)
    return merged

# Columns (1) to (3) use the complete-data population: exposed and undefended,
# with known roof class, vegetation and construction year. Column (4) restores
# defended structures and carries an indicator for them.
regression_sample = _prepare_sample(samples.complete)
defense_sample = _prepare_sample(
    samples.exposed[samples.exposed.roof_class.ne('Unknown')]
    .dropna(subset=['ndvi_iqr', 'year_decade'])
)

# Continuous predictors are centered at the estimation-sample means. Slopes are
# unchanged, so the proportional readings still hold, but the main effects of
# the interacted model are then evaluated at mean exposure rather than at
# F* = 1, which lies outside the data. Column (4) uses the same constants so
# its coefficients remain comparable.
CENTRING = {column: float(regression_sample[column].mean())
            for column in ['log_F_star', 'log_ndvi', 'log_influence_depth',
                           'year_decade', 'log_n_visible', 'log_footprint']}
for column, constant in CENTRING.items():
    regression_sample[column] = regression_sample[column] - constant
    defense_sample[column] = defense_sample[column] - constant

# Escalation conditions on having sustained damage, so it is estimated on the
# damaged subset of the same structures, with the same centering.
escalation_sample = regression_sample[regression_sample.any_damage.eq(1)].copy()

SITE_TERM = 'log_n_visible'
SIZE_TERM = 'log_footprint'
BASE_TERMS = (f'log_F_star + {ROOF_TERM} + log_ndvi + year_decade '
              f'+ {SITE_TERM} + {SIZE_TERM} + C(fire)')
DEPTH_TERM = 'log_influence_depth'
# Model (0) drops realized exposure. F* lies on the path from site position to
# destruction, so (0) carries the total association of position with the
# outcome while (1) reports it at equal exposure.
NO_EXPOSURE_TERMS = (f'{ROOF_TERM} + log_ndvi + year_decade '
                     f'+ {SITE_TERM} + {SIZE_TERM} + C(fire)')
INTERACTION_TERMS = (f'log_F_star:({ROOF_TERM} + log_ndvi + year_decade '
                     f'+ log_influence_depth + {SITE_TERM})')
MODEL_SPECIFICATIONS = [
    ('no_exposure', 'Destruction, no exposure control', 'is_destroyed',
     'Destroyed', f'{NO_EXPOSURE_TERMS} + {DEPTH_TERM}', regression_sample,
     'Undefended'),
    ('site_depth', 'Destruction', 'is_destroyed', 'Destroyed',
     f'{BASE_TERMS} + {DEPTH_TERM}', regression_sample, 'Undefended'),
    ('any_damage', 'Any damage', 'any_damage', 'Any damage',
     f'{BASE_TERMS} + {DEPTH_TERM}', regression_sample, 'Undefended'),
    ('escalation', 'Destruction given damage', 'is_destroyed',
     'Destroyed, given damage', f'{BASE_TERMS} + {DEPTH_TERM}',
     escalation_sample,
     'Undefended, damaged'),
    ('exposure_interactions', '+ Exposure interactions', 'is_destroyed',
     'Destroyed',
     f'{BASE_TERMS} + {DEPTH_TERM} + {INTERACTION_TERMS}',
     regression_sample, 'Undefended'),
    ('documented_defense', '+ Documented defense', 'is_destroyed',
     'Destroyed', f'{BASE_TERMS} + {DEPTH_TERM} + defended_structure',
     defense_sample, 'Defended retained'),
]
FITTED_MODELS = [
    (model_key, model_label, outcome_label,
     fit_clustered_logit(sample, f'{outcome} ~ {terms}'))
    for model_key, model_label, outcome, outcome_label, terms, sample, _
    in MODEL_SPECIFICATIONS
]
MODEL_SAMPLES = {model_key: (sample, sample_label)
                 for model_key, _, _, _, _, sample, sample_label
                 in MODEL_SPECIFICATIONS}

TERM_LABELS = {
    'Intercept': 'Intercept',
    'log_F_star': 'Realized coupling, log F*',
    'log_ndvi': 'Log near-structure NDVI',
    'log_influence_depth': 'Log distance into built environment',
    'log_n_visible': 'Log connected neighbors (pre-fire)',
    'log_footprint': 'Log footprint area',
    'year_decade': 'Building vintage, per decade newer',
    'defended_structure': 'Documented property defense',
    'defended_structure': 'Documented property defense',
    'C(fire)[T.PALISADES]': 'Fire: Palisades',
}


def _term_label(term: str) -> str:
    """Render a patsy term as prose, splitting interactions on the colon."""
    pieces = term.split(':')
    parts = []
    for part in pieces:
        if part in TERM_LABELS:
            # Inside an interaction the exposure term needs only its symbol.
            parts.append('log F*' if len(pieces) > 1 and part == 'log_F_star'
                         else TERM_LABELS[part])
        elif part.startswith('C(roof_class'):
            parts.append(f'{part.split("[T.")[1][:-1]} roof')
        else:
            parts.append(part)
    return ' × '.join(parts)


coefficient_rows, diagnostic_rows = [], []
for model_key, model_label, outcome, model in FITTED_MODELS:
    confidence = model.conf_int()
    for term in model.params.index:
        coefficient_rows.append({
            'model': model_key, 'model_label': model_label,
            'outcome': outcome, 'term': term, 'term_label': _term_label(term),
            'coefficient': float(model.params[term]),
            'cluster_robust_se': float(model.bse[term]),
            'z': float(model.tvalues[term]),
            'p_value': float(model.pvalues[term]),
            'ci_lo': float(confidence.loc[term, 0]),
            'ci_hi': float(confidence.loc[term, 1]),
            'odds_ratio': float(np.exp(model.params[term])),
            'odds_ratio_ci_lo': float(np.exp(confidence.loc[term, 0])),
            'odds_ratio_ci_hi': float(np.exp(confidence.loc[term, 1])),
        })
    sample, sample_label = MODEL_SAMPLES[model_key]
    diagnostic_rows.append({
        'model': model_key, 'model_label': model_label, 'outcome': outcome,
        'sample': sample_label,
        'defended_structures': int(sample.defended_structure.sum()),
        'observations': int(model.nobs),
        'events': int(np.asarray(model.model.endog).sum()),
        'grid_clusters': int(pd.unique(model.cov_kwds['groups']).size),
        'parameters': int(model.df_model) + 1,
        'log_likelihood': float(model.llf),
        'log_likelihood_null': float(model.llnull),
        'mcfadden_pseudo_r2': float(model.prsquared),
        'aic': float(model.aic),
        'converged': bool(model.mle_retvals['converged']),
        'covariance': 'cluster-robust on 250 m grid_id',
    })
fitted_model_coefficients = pd.DataFrame(coefficient_rows)
fitted_model_diagnostics = pd.DataFrame(diagnostic_rows)

# Joint Wald tests on the added blocks, using the cluster-robust covariance
# rather than a likelihood-ratio test, which clustering would invalidate.
wald_rows = []
for model_key, block_label, block_terms in [
        ('exposure_interactions', 'F* interactions', None),
        ('exposure_interactions', 'Connected neighbors',
         ['log_n_visible', 'log_F_star:log_n_visible']),
        ('documented_defense', 'Documented defense',
         ['defended_structure'])]:
    model = dict((key, fitted) for key, _, _, fitted in FITTED_MODELS)[model_key]
    terms = (block_terms if block_terms is not None
             else [term for term in model.params.index if ':' in term])
    restriction = np.zeros((len(terms), len(model.params)))
    for row, term in enumerate(terms):
        restriction[row, list(model.params.index).index(term)] = 1
    test = model.wald_test(restriction, scalar=True)
    wald_rows.append({
        'model': model_key, 'block': block_label, 'terms': len(terms),
        'chi2': float(test.statistic), 'p_value': float(test.pvalue),
    })
block_wald_tests = pd.DataFrame(wald_rows)

fitted_model_coefficients.to_csv(
    RESULTS / 'tableS_construction_fitted_models.csv', index=False)
fitted_model_diagnostics.to_csv(
    RESULTS / 'tableS_construction_model_diagnostics.csv', index=False)
block_wald_tests.to_csv(
    RESULTS / 'tableS_construction_block_wald_tests.csv', index=False)

summary_blocks = []
for model_key, model_label, outcome, model in FITTED_MODELS:
    heading = f'{model_label} — outcome: {outcome} [{model_key}]'
    summary_blocks.append(
        f'{heading}\n{"=" * len(heading)}\n{model.summary().as_text()}')
model_summaries_text = '\n\n'.join(summary_blocks)
(RESULTS / 'tableS_construction_model_summaries.txt').write_text(
    model_summaries_text + '\n', encoding='utf-8')

print('Centring constants (subtracted before fitting): '
      + ', '.join(f'{name} {value:.4f}' for name, value in CENTRING.items()))
print(model_summaries_text)
display(fitted_model_diagnostics.round(3))
display(block_wald_tests.round(4))
print('saved', RESULTS / 'tableS_construction_fitted_models.csv')
print('saved', RESULTS / 'tableS_construction_model_summaries.txt')

### Supplementary regression table

*Supplement — Supplementary Table S2.*

The fitted models are rendered as a single supplementary table. Columns (1) to
(3) are the three outcome models that Table 1 summarizes as
exposure-tolerance ratios, and are its raw-coefficient companion; columns (4)
and (5) are the two extensions, the exposure-interaction model supporting the
exposure-dependence result and the standardized margins, and the
documented-defense model that the defense section builds on. Cells are logit
coefficients with cluster-robust standard errors, followed by a proportional
restatement of the logged terms and the sample and fit diagnostics for each
column.

In [ ]:
ROOF_ORDER = ['Tile', 'Metal', 'Concrete', 'Combustible', 'Other']
EQUIVALENT_PERCENT = 10  # proportional step used for the logged predictors
LOGGED_TERMS = ['log_F_star', 'log_ndvi', 'log_influence_depth',
                'log_n_visible', 'log_footprint']
TERM_BLOCKS = [
    # F* is realized during the fire, unlike the pre-fire characteristics.
    ('Realized exposure', ['log_F_star']),
    ('Property and site characteristics',
     [f'{ROOF_TERM}[T.{level}]' for level in ROOF_ORDER]
     + ['log_ndvi', 'year_decade', 'log_influence_depth', 'log_n_visible',
        'log_footprint', 'defended_structure']),
    ('Exposure × property characteristics', None),  # every interaction term
    ('Controls', ['C(fire)[T.PALISADES]', 'Intercept']),
]
# All six fits are reported in one supplementary table: column (0) drops the
# exposure control, columns (1) to (3) are the models Table 1 summarizes, and
# columns (4) and (5) are the two extensions used elsewhere in the text.
# Numbering starts at zero so the column references elsewhere still hold.
MODEL_GROUPS = [
    ('regression_table',
     ['no_exposure', 'site_depth', 'any_damage', 'escalation',
      'exposure_interactions', 'documented_defense'],
     '**Supplementary Table | Logistic regressions of structure outcomes.**',
     0),
]


def _stars(p_value: float) -> str:
    if p_value < .001:
        return '***'
    if p_value < .01:
        return '**'
    if p_value < .05:
        return '*'
    return ''


def _coefficient_cell(row) -> str:
    return (f'{row.coefficient:.3f}{_stars(row.p_value)} '
            f'({row.cluster_robust_se:.3f})').replace('-', '−')


def _p_text(value: float) -> str:
    return '< 0.001' if value < .001 else f'= {value:.3f}'


term_labels = fitted_model_coefficients.set_index('term').term_label.to_dict()
diagnostics = fitted_model_diagnostics.set_index('model')
wald = block_wald_tests.set_index('block')
equivalent_scale = np.log1p(EQUIVALENT_PERCENT / 100)


def _render_group(model_keys: list[str], title: str,
                  start_index: int = 1) -> tuple[pd.DataFrame, str]:
    """Render one supplementary regression table for a group of models."""
    labels = [diagnostics.loc[key, 'model_label'] for key in model_keys]
    cells = (fitted_model_coefficients[
        fitted_model_coefficients.model.isin(model_keys)]
        .assign(cell=lambda frame: frame.apply(_coefficient_cell, axis=1))
        .pivot(index='term', columns='model_label', values='cell')
        .reindex(columns=labels).fillna('—'))
    interactions = [term for term in cells.index if ':' in term]
    rows = []
    for block_label, block_terms in TERM_BLOCKS:
        terms = interactions if block_terms is None else block_terms
        terms = [term for term in terms if term in cells.index]
        if not terms:
            continue
        rows.append({'Term': f'**{block_label}**',
                     **{label: '' for label in labels}})
        for term in terms:
            rows.append({'Term': term_labels[term],
                         **cells.loc[term].to_dict()})
    logged = [term for term in LOGGED_TERMS if term in cells.index]
    if logged:
        rows.append({'Term': f'**Odds change per {EQUIVALENT_PERCENT}% '
                             'increase**', **{label: '' for label in labels}})
        for term in logged:
            coefficients = (
                fitted_model_coefficients[
                    fitted_model_coefficients.term.eq(term)
                    & fitted_model_coefficients.model.isin(model_keys)]
                .set_index('model_label').coefficient)
            row = {}
            for label in labels:
                if label not in coefficients.index:
                    row[label] = '—'
                    continue
                change = 100 * (np.exp(coefficients[label]
                                       * equivalent_scale) - 1)
                row[label] = f'{change:+.1f}%'.replace('-', '−')
            rows.append({'Term': term_labels[term], **row})
    for label, column, fmt in [('Outcome', 'outcome', '{}'),
                               ('Sample', 'sample', '{}'),
                               ('Structures', 'observations', '{:,.0f}'),
                               ('Events', 'events', '{:,.0f}'),
                               ('Grid clusters', 'grid_clusters', '{:,.0f}'),
                               ('McFadden pseudo-R²', 'mcfadden_pseudo_r2',
                                '{:.3f}')]:
        rows.append({'Term': f'**{label}**',
                     **{diagnostics.loc[key, 'model_label']:
                        fmt.format(diagnostics.loc[key, column])
                        for key in model_keys}})
    table = pd.DataFrame(rows).set_index('Term')
    lines = [title, '', '| Term | ' + ' | '.join(
        f'({index}) {label}'
        for index, label in enumerate(labels, start_index)) + ' |',
        '|---|' + '|'.join([':--:'] * len(labels)) + '|']
    for term, row in table.iterrows():
        lines.append(f'| {term} | ' + ' | '.join(row) + ' |')
    return table, '\n'.join(lines)


SHARED_NOTE = (
    'Logistic regressions of structure outcomes. Coefficients are log-odds '
    'estimates; standard errors in parentheses are clustered on the 250 m '
    'analysis grid cell, which is not a term in the model. Reference '
    'categories are asphalt roof and Eaton. Continuous predictors are centered '
    'at their estimation-sample means. *F*\\* is the view-factor-weighted '
    'realized coupling from neighboring structures that were themselves '
    'destroyed and is realized during the fire; connected neighbors counts '
    'every structure with a visible view of the focal building, burned or '
    'not, and is fixed pre-fire geometry. Distance into the built environment '
    'is the distance to the cleaned, fire-clipped CAL FIRE WUI Influence Zone '
    'boundary. NDVI is a pre-fire Sentinel-2 composite (October–December '
    f'2024). Rows under \'odds change per {EQUIVALENT_PERCENT}% increase\' '
    'restate the logged coefficients as the multiplicative change in the odds '
    f'of the outcome for a {EQUIVALENT_PERCENT}% increase in that predictor. '
    '\\*\\*\\**P* < 0.001, \\*\\**P* < 0.01, \\**P* < 0.05.')
GROUP_NOTES = {
    'regression_table': (
        'Column (0) is the column (1) specification without log *F*\\*. '
        'Realized coupling lies on the path from site position to '
        'destruction, so column (1) reports associations at equal exposure '
        'while column (0) reports the total association; the two differ in '
        'sign for distance into the built environment and for connected '
        'neighbors, both of which predict exposure itself. '
        'Columns (1) to (3) are the fits summarized as exposure-tolerance '
        'ratios in Table 1: columns (1) and (2) share a sample and differ only '
        'in the outcome, and column (3) conditions on having sustained damage. '
        'Structures with documented defense are excluded from those columns '
        'rather than adjusted for, because suppression is deployed in response '
        'to fire behavior and is therefore a consequence of exposure. Column '
        '(4) adds interactions between log *F*\\* and each characteristic, '
        'which are jointly significant on the cluster-robust covariance '
        f'($\\chi^2$({wald.loc["F* interactions", "terms"]:.0f}) = '
        f'{wald.loc["F* interactions", "chi2"]:.1f}, '
        f'*P* {_p_text(wald.loc["F* interactions", "p_value"])}); because the '
        'predictors are centered, its main effects are read at mean covariate '
        'values rather than at zero. Column (5) restores the defended '
        'structures and carries an indicator for them '
        f'($\\chi^2$({wald.loc["Documented defense", "terms"]:.0f}) = '
        f'{wald.loc["Documented defense", "chi2"]:.1f}, '
        f'*P* {_p_text(wald.loc["Documented defense", "p_value"])}), and is '
        'the specification the defense section builds on. '),
}

supplementary_regression_tables = {}
for group_key, model_keys, title, start_index in MODEL_GROUPS:
    table, markdown = _render_group(model_keys, title, start_index)
    markdown = '\n'.join([markdown, '', GROUP_NOTES[group_key] + SHARED_NOTE])
    supplementary_regression_tables[group_key] = table
    table.to_csv(RESULTS / f'tableS_construction_{group_key}.csv')
    (RESULTS / f'tableS_construction_{group_key}.md').write_text(
        markdown + '\n', encoding='utf-8')
    display(Markdown(markdown))
print('saved', RESULTS / 'tableS_construction_regression_table.md')

### Main table

**Main text — Table 1.**

The supplementary regression table reports log-odds coefficients for every
term of every model, which is the right form for an audit trail and the wrong
form for the main article. This cell renders the same fits as a compact main
table: five attributes in rows, the three outcomes in columns, and exposure
tolerance ratios rather than coefficients, matching the estimand the results
section quotes. Realized coupling has no tolerance ratio of its own, since it
is the scale the ratios are measured against, so its coefficient is stated in
the legend instead of occupying a row on a different scale.

The legend carries what Nature journals require of a statistical table: the
model class and outcome, the estimand and how it and its interval are
computed, the error structure and unit of clustering, every covariate adjusted
for, the span each contrast is reported over with its units, exact two-sided *P* values rather than thresholds, the analyzed sample and event count per
column, the conditioning of the third column with the number of structures
damaged but not destroyed, a pointer to the complete regression output, and
software versions.

In [ ]:
import statsmodels

from src.analysis.construction import tolerance_ratio

_fitted = dict((key, model) for key, _, _, model in FITTED_MODELS)
MAIN_OUTCOMES = [('site_depth', 'Destruction'),
                 ('any_damage', 'Any damage'),
                 ('escalation', 'Destruction given damage')]


def _iqr_span(column: str) -> tuple[float, float, float]:
    """Interquartile span of a centered log predictor, and its raw quartiles."""
    low, high = np.percentile(regression_sample[column], [25, 75])
    raw = np.exp(np.array([low, high]) + CENTRING[column])
    return float(high - low), float(raw[0]), float(raw[1])


ndvi_span, _, _ = _iqr_span('log_ndvi')
depth_span, depth_q1, depth_q3 = _iqr_span('log_influence_depth')
connected_span, connected_q1, connected_q3 = _iqr_span('log_n_visible')
footprint_span, footprint_q1, footprint_q3 = _iqr_span('log_footprint')
MAIN_TERMS = [
    (f'{ROOF_TERM}[T.Tile]', 'Tile roof, versus asphalt', 1.),
    ('log_ndvi', 'Near-structure vegetation, per interquartile increase in '
     'NDVI', ndvi_span),
    ('year_decade', 'Building vintage, per decade newer', 1.),
    ('log_influence_depth',
     f'Distance into urban core, {depth_q1:,.0f} versus {depth_q3:,.0f} ft',
     depth_span),
    ('log_n_visible',
     f'Connected neighbors, {connected_q1:,.0f} versus {connected_q3:,.0f}',
     connected_span),
    ('log_footprint',
     f'Footprint area, {footprint_q1:,.0f} versus {footprint_q3:,.0f} m²',
     footprint_span),
]


def _p_text(value: float) -> str:
    """Exact P value, as Nature journals ask for rather than a threshold."""
    if value >= .001:
        return f'= {value:.3f}'
    if value <= 0:  # underflowed to zero in double precision
        return '< 1 × 10^−300^'
    exponent = int(np.floor(np.log10(value)))
    return f'= {value / 10 ** exponent:.1f} × 10^{exponent}^'


main_rows, main_records = [], []
for term, label, span in MAIN_TERMS:
    row = {'Term': label}
    for key, outcome_label in MAIN_OUTCOMES:
        model = _fitted[key]
        ratio = tolerance_ratio(model, term)
        log_ratio = np.log(ratio['tolerance_ratio'])
        log_se = (np.log(ratio['ci_hi']) - np.log(ratio['ci_lo'])) / (2 * 1.96)
        estimate = np.exp(log_ratio * span)
        low = np.exp((log_ratio - 1.96 * log_se) * span)
        high = np.exp((log_ratio + 1.96 * log_se) * span)
        row[outcome_label] = (f'{estimate:.2f} ({low:.2f}–{high:.2f}), '
                              f'*P* {_p_text(ratio["p_value"])}')
        main_records.append({
            'term': term, 'term_label': label, 'outcome': outcome_label,
            'tolerance_ratio': float(estimate), 'ci_lo': float(low),
            'ci_hi': float(high), 'p_value': ratio['p_value'],
            'coefficient': float(model.params[term]),
            'cluster_robust_se': float(model.bse[term]),
            'reported_span_in_log_units': span,
        })
    main_rows.append(row)
construction_main_records = pd.DataFrame(main_records)
construction_main_table = pd.DataFrame(main_rows).set_index('Term')

diagnostics = fitted_model_diagnostics.set_index('model')
summary_rows = [
    ('Structures analyzed', 'observations', '{:,.0f}'),
    ('Outcome events', 'events', '{:,.0f}'),
    ('Grid clusters', 'grid_clusters', '{:,.0f}'),
]
for label, column, fmt in summary_rows:
    row = {'Term': label}
    for key, outcome_label in MAIN_OUTCOMES:
        row[outcome_label] = fmt.format(diagnostics.loc[key, column])
    main_rows.append(row)
construction_main_display = pd.DataFrame(main_rows).set_index('Term')
construction_main_display.to_csv(RESULTS / 'table1_construction_outcomes.csv')
construction_main_records.to_csv(
    RESULTS / 'table1_construction_outcomes_estimates.csv', index=False)

columns = [outcome_label for _, outcome_label in MAIN_OUTCOMES]
header = '| Term | ' + ' | '.join(columns) + ' |'
damaged_only = (int(diagnostics.loc['escalation', 'observations'])
                - int(diagnostics.loc['escalation', 'events']))
legend = (
    'Multivariable logistic regressions on '
    f'{diagnostics.loc["site_depth", "observations"]:,.0f} exposed, '
    'undefended structures with known roof class, vegetation and construction '
    'year. Entries are exposure-tolerance ratios, exp(−*γ*/*β*): the multiple '
    'of realized coupling *F*\\* giving the same fitted outcome probability, '
    'where *γ* is the attribute coefficient and *β* that on log *F*\\*; '
    'values above one indicate greater tolerance. *F*\\* itself has no ratio, '
    'being the scale (*β* = '
    + ', '.join(f'{_fitted[key].params["log_F_star"]:.2f}'
                for key, _ in MAIN_OUTCOMES)
    + ' across columns). Delta-method 95% confidence intervals; Wald *P* '
    'values floored at 0.001, exact values in Source Data; standard errors '
    'clustered on the 250 m analysis grid cell. All columns adjust for log '
    '*F*\\*, roof class (asphalt reference), log NDVI, construction decade, '
    'log distance into the urban core, log connected neighbors and fire. '
    'Logged predictors are reported over their interquartile range. Column 3 '
    f'conditions on having sustained damage; {damaged_only:,} damaged '
    'structures were not destroyed. Full model output: Supplementary Table '
    'S2.')

lines = ['**Table 1 | Structure outcomes as a function of realized radiative '
         'coupling and property characteristics.**', '', header,
         '|---|' + '|'.join([':--:'] * len(columns)) + '|']
for term, row in construction_main_display.iterrows():
    lines.append(f'| {term} | ' + ' | '.join(row[c] for c in columns) + ' |')
lines += ['', legend]
construction_main_markdown = '\n'.join(lines)
(RESULTS / 'table1_construction_outcomes.md').write_text(
    construction_main_markdown + '\n', encoding='utf-8')

display(Markdown(construction_main_markdown))
print('saved', RESULTS / 'table1_construction_outcomes.md')

### Supplementary table — average marginal effects

*Supplement — Supplementary Table S5.*

Log-odds coefficients are identified only relative to the residual variance of
their own model, so a coefficient in column (3), fitted to a different outcome
on a different sample, is not on the same scale as columns (1) and (2), and a
gap between them need not reflect a difference in association. This cell
restates columns (1) to (3) as average marginal effects: the change in outcome
probability, in percentage points, averaged over each estimation sample, which
does share a scale across columns. Each effect is a counterfactual contrast.
Every structure is predicted twice with one predictor changed, and the two
predictions are averaged and differenced. Logged predictors are reported over
two spans: a 10% increase, the marginal reading, and a move from the first to
the third quartile of the column (1) sample, the span Table 1 uses and the
scale on which the results section compares attributes. Vintage is advanced
one decade and each roof class is set against asphalt for all structures.
Standard errors use the delta method on the cluster-robust covariance.

In [ ]:
from patsy import build_design_matrices
from scipy.special import expit
from scipy.stats import norm

# Average marginal effects: the change in the probability of the outcome, in
# percentage points, averaged over each model's estimation sample. Log-odds
# coefficients are scaled by the latent residual variance of their own model,
# so they cannot be compared across outcomes or samples; probabilities can.
# Each effect is a counterfactual contrast: every structure is predicted twice
# with one predictor changed and the rest held at their observed values, and
# the mean difference is reported. Standard errors use the delta method on the
# cluster-robust covariance.
AME_PERCENT = 10
AME_MODELS = [('site_depth', 'Destruction'),
              ('any_damage', 'Any damage'),
              ('escalation', 'Destruction given damage')]
AME_STEP = np.log1p(AME_PERCENT / 100)
AME_LOGGED = [
    ('log_F_star', 'Realized coupling, F*', '{:.3f}'),
    ('log_ndvi', 'Near-structure NDVI', '{:.2f}'),
    ('log_influence_depth', 'Distance into built environment', '{:,.0f} ft'),
    ('log_n_visible', 'Connected neighbors (pre-fire)', '{:,.0f}'),
    ('log_footprint', 'Footprint area', '{:,.0f} m²'),
]
# Quartiles come from the column (1) sample, as in Table 1, so every column is
# evaluated over the same contrast, including the damaged subset of column (3).
AME_QUARTILES = {column: tuple(np.percentile(regression_sample[column], [25, 75]))
                 for column, _, _ in AME_LOGGED}


def _quartile_label(column: str, label: str, form: str) -> str:
    low, high = np.exp(np.array(AME_QUARTILES[column]) + CENTRING[column])
    return f'{label}, {form.format(low)} to {form.format(high)}'


AME_CONTRASTS = (
    [(f'Per {AME_PERCENT}% increase', label, 'shift', column, AME_STEP)
     for column, label, _ in AME_LOGGED]
    + [('First to third quartile', _quartile_label(column, label, form),
        'quartiles', column, AME_QUARTILES[column])
       for column, label, form in AME_LOGGED]
    + [('Roof material, versus asphalt', level, 'roof', 'roof_class', level)
       for level in ROOF_ORDER]
    + [('Building vintage', 'One decade newer', 'shift', 'year_decade', 1.)]
)


def _average_marginal_effect(model, baseline: pd.DataFrame,
                             counterfactual: pd.DataFrame) -> tuple[float, float]:
    """Mean change in predicted probability, with a delta-method SE."""
    design_info = model.model.data.design_info
    params = model.params.to_numpy()
    before, after = (np.asarray(build_design_matrices([design_info], frame)[0])
                     for frame in (baseline, counterfactual))
    p_before, p_after = expit(before @ params), expit(after @ params)
    gradient = ((p_after * (1 - p_after))[:, None] * after
                - (p_before * (1 - p_before))[:, None] * before).mean(axis=0)
    standard_error = float(np.sqrt(
        gradient @ model.cov_params().to_numpy() @ gradient))
    return float(np.mean(p_after - p_before)), standard_error


fitted_by_key = dict((key, model) for key, _, _, model in FITTED_MODELS)
ame_records = []
for model_key, outcome_label in AME_MODELS:
    model = fitted_by_key[model_key]
    # Only the rows the model was fitted on, so the average matches its sample.
    sample = MODEL_SAMPLES[model_key][0].loc[model.model.data.row_labels]
    for block, label, kind, column, value in AME_CONTRASTS:
        if kind == 'roof':
            baseline = sample.assign(roof_class='Asphalt')
            counterfactual = sample.assign(roof_class=value)
        elif kind == 'quartiles':
            baseline = sample.assign(**{column: value[0]})
            counterfactual = sample.assign(**{column: value[1]})
        else:
            baseline = sample
            counterfactual = sample.assign(**{column: sample[column] + value})
        effect, standard_error = _average_marginal_effect(
            model, baseline, counterfactual)
        ame_records.append({
            'model': model_key, 'outcome': outcome_label, 'block': block,
            'contrast': label,
            'ame_pp': 100 * effect, 'se_pp': 100 * standard_error,
            'ci_lo_pp': 100 * (effect - 1.96 * standard_error),
            'ci_hi_pp': 100 * (effect + 1.96 * standard_error),
            'p_value': float(2 * norm.sf(abs(effect / standard_error))),
            'outcome_rate_percent': 100 * float(model.model.endog.mean()),
            'observations': int(model.nobs),
        })
average_marginal_effects = pd.DataFrame(ame_records)
average_marginal_effects.to_csv(
    RESULTS / 'tableS_construction_average_marginal_effects.csv', index=False)


def _ame_cell(row) -> str:
    return (f'{row.ame_pp:+.2f}{_stars(row.p_value)} '
            f'({row.ci_lo_pp:.2f} to {row.ci_hi_pp:.2f})').replace('-', '−')


ame_columns = [outcome_label for _, outcome_label in AME_MODELS]
ame_cells = (average_marginal_effects
             .assign(cell=lambda frame: frame.apply(_ame_cell, axis=1))
             .set_index(['block', 'contrast', 'outcome']).cell)
ame_rates = average_marginal_effects.groupby('outcome').first()
ame_rows = []
for block in dict.fromkeys(block for block, _, _, _, _ in AME_CONTRASTS):
    ame_rows.append(f'| **{block}** |' + ' |' * len(ame_columns))
    for _, label, _, _, _ in [c for c in AME_CONTRASTS if c[0] == block]:
        ame_rows.append(f'| {label} | ' + ' | '.join(
            ame_cells[(block, label, outcome)] for outcome in ame_columns)
            + ' |')
ame_markdown_lines = [
    '**Supplementary Table | Average marginal effects on the probability of '
    'structure outcomes.**',
    '',
    '| Contrast | ' + ' | '.join(
        f'({index}) {label}' for index, label in enumerate(ame_columns, 1))
    + ' |',
    '|---|' + ':--:|' * len(ame_columns),
    *ame_rows,
    '| **Outcome rate** | ' + ' | '.join(
        f'{ame_rates.loc[label, "outcome_rate_percent"]:.1f}%'
        for label in ame_columns) + ' |',
    '| **Structures** | ' + ' | '.join(
        f'{ame_rates.loc[label, "observations"]:,}' for label in ame_columns)
    + ' |',
    '',
    'Average marginal effects from columns (1) to (3) of the logistic '
    'regression table, in percentage points of outcome probability. Every '
    'structure in the estimation sample is predicted twice with one predictor '
    'changed and every other covariate held at its observed value, and the '
    f'mean difference is reported. Logged predictors are increased by '
    f'{AME_PERCENT}% on their original scale, or set to their first and then '
    'third quartile in the column (1) sample, so every column is evaluated '
    'over the same contrast; construction year is advanced one decade; roof '
    'material is set to the named class rather than asphalt for all '
    'structures. Unlike log-odds coefficients, these effects share a scale '
    'across columns; effects in column (3) are bounded by its high outcome '
    'rate. 95% confidence intervals use the delta method on the covariance '
    'clustered on the 250 m analysis grid cell. '
    '\\*\\*\\**P* < 0.001, \\*\\**P* < 0.01, \\**P* < 0.05.',
]
ame_markdown = '\n'.join(ame_markdown_lines)
(RESULTS / 'tableS_construction_average_marginal_effects.md').write_text(
    ame_markdown + '\n', encoding='utf-8')
display(Markdown(ame_markdown))
print('saved', RESULTS / 'tableS_construction_average_marginal_effects.csv')
print('saved', RESULTS / 'tableS_construction_average_marginal_effects.md')

### Supplementary table — building-code eras

*Supplement — Supplementary Table S?.*

California tightened construction standards for fire-prone areas twice: the
1992 Bates Bill set minimum roof and fire standards in Very High Fire Hazard
Severity Zones, and Chapter 7A of the California Building Code set WUI
construction standards for permits from 2008. This cell replaces continuous
vintage with three eras (before 1992, 1992–2008, after 2008) in each of the
three outcome models and reports each era against pre-1992 construction at
equal exposure, as an average marginal effect and as an exposure-tolerance
ratio. Four specifications separate what the eras capture: era alone, which
counts code-required roof upgrades toward the era; adding roof material;
adding a linear vintage trend, so each era term is a step at code adoption
rather than newer construction in general; and Palisades alone, where the
assessed structures lie in Very High hazard zones and both codes applied.
Era comes from the assessor construction year, which does not record
retrofits or re-roofing, and only 470 structures postdate 2008 (50 in Eaton),
so the post-2008 estimates are imprecise.

In [ ]:
from src.analysis.construction import tolerance_ratio

# Building-code eras: the 1992 Bates Bill (minimum roof and fire standards in
# Very High Fire Hazard Severity Zones) and Chapter 7A of the California
# Building Code (WUI construction standards, effective for permits from 2008).
# Construction year replaces continuous vintage with three eras, so each
# coefficient compares an era with pre-1992 construction at equal exposure.
ERA_TERM = 'C(code_era, Treatment("Pre-1992"))'
ERA_LEVELS = ['1992-2008', 'Post-2008']
ERA_LABELS = {'1992-2008': '1992–2008 (Bates Bill)',
              'Post-2008': 'Post-2008 (Chapter 7A)'}
ERA_CONTROLS = ('log_F_star + log_ndvi + log_influence_depth '
                f'+ {SITE_TERM} + {SIZE_TERM}')
# (1) era alone keeps roof upgrades, which the codes required, inside the era
# effect; (2) adds roof material, leaving what the codes changed beyond the
# roof; (3) adds a linear vintage trend, so each era term is a step at code
# adoption rather than newer construction generally; (4) repeats (2) in
# Palisades, where the assessed structures lie in Very High hazard zones and
# both codes applied.
ERA_SPECIFICATIONS = [
    ('era', '(1) Era only', f'{ERA_TERM} + {ERA_CONTROLS} + C(fire)', None),
    ('era_roof', '(2) + Roof material',
     f'{ERA_TERM} + {ROOF_TERM} + {ERA_CONTROLS} + C(fire)', None),
    ('era_roof_trend', '(3) + Linear vintage trend',
     f'{ERA_TERM} + {ROOF_TERM} + year_decade + {ERA_CONTROLS} + C(fire)',
     None),
    ('era_roof_palisades', '(4) Palisades only, as (2)',
     f'{ERA_TERM} + {ROOF_TERM} + {ERA_CONTROLS}', 'PALISADES'),
]
ERA_OUTCOMES = [('is_destroyed', 'Destruction', regression_sample),
                ('any_damage', 'Any damage', regression_sample),
                ('is_destroyed', 'Destruction given damage', escalation_sample)]

era_counts = (regression_sample
              .assign(code_era=regression_sample.age_cohort.astype(str),
                      tile=regression_sample.roof_class.eq('Tile'))
              .groupby(['fire', 'code_era'])
              .agg(structures=('is_destroyed', 'size'),
                   destroyed_share=('is_destroyed', 'mean'),
                   damaged_share=('any_damage', 'mean'),
                   tile_share=('tile', 'mean'))
              .reset_index())

era_rows, era_tests = [], []
for spec_key, spec_label, terms, fire_only in ERA_SPECIFICATIONS:
    for outcome, outcome_label, base_sample in ERA_OUTCOMES:
        sample = base_sample.assign(
            code_era=base_sample.age_cohort.astype(str))
        if fire_only is not None:
            sample = sample[sample.fire.eq(fire_only)]
        model = fit_clustered_logit(sample, f'{outcome} ~ {terms}')
        fitted = sample.loc[model.model.data.row_labels]
        era_terms = [f'{ERA_TERM}[T.{level}]' for level in ERA_LEVELS]
        restriction = np.zeros((len(era_terms), len(model.params)))
        for row, term in enumerate(era_terms):
            restriction[row, list(model.params.index).index(term)] = 1
        joint = model.wald_test(restriction, scalar=True)
        era_tests.append({'specification': spec_label,
                          'outcome': outcome_label,
                          'chi2': float(joint.statistic),
                          'p_value': float(joint.pvalue)})
        for level, term in zip(ERA_LEVELS, era_terms):
            effect, standard_error = _average_marginal_effect(
                model, fitted.assign(code_era='Pre-1992'),
                fitted.assign(code_era=level))
            ratio = tolerance_ratio(model, term)
            era_rows.append({
                'specification': spec_label, 'outcome': outcome_label,
                'code_era': level, 'era_label': ERA_LABELS[level],
                'structures_in_era': int(fitted.code_era.eq(level).sum()),
                'observations': int(model.nobs),
                'coefficient': float(model.params[term]),
                'cluster_robust_se': float(model.bse[term]),
                'p_value': float(model.pvalues[term]),
                'ame_pp': 100 * effect, 'ame_se_pp': 100 * standard_error,
                'ame_ci_lo_pp': 100 * (effect - 1.96 * standard_error),
                'ame_ci_hi_pp': 100 * (effect + 1.96 * standard_error),
                'tolerance_ratio': ratio['tolerance_ratio'],
                'tolerance_ci_lo': ratio['ci_lo'],
                'tolerance_ci_hi': ratio['ci_hi'],
            })
code_era_effects = pd.DataFrame(era_rows)
code_era_tests = pd.DataFrame(era_tests)
era_counts.to_csv(RESULTS / 'tableS_code_era_counts.csv', index=False)
code_era_effects.to_csv(RESULTS / 'tableS_code_era_effects.csv', index=False)
code_era_tests.to_csv(RESULTS / 'tableS_code_era_joint_tests.csv',
                      index=False)


def _era_cell(row) -> str:
    return (f'{row.ame_pp:+.1f}{_stars(row.p_value)} '
            f'({row.ame_ci_lo_pp:.1f} to {row.ame_ci_hi_pp:.1f}); '
            f'{row.tolerance_ratio:.2f}').replace('-', '−')


era_outcomes = [label for _, label, _ in ERA_OUTCOMES]
era_cells = (code_era_effects
             .assign(cell=lambda frame: frame.apply(_era_cell, axis=1))
             .set_index(['specification', 'era_label', 'outcome']).cell)
era_joint = code_era_tests.set_index(['specification', 'outcome']).p_value
era_markdown_lines = [
    '**Supplementary Table | Destruction by building-code era.**',
    '',
    '| Specification | Era, versus pre-1992 | ' + ' | '.join(
        f'({index}) {label}' for index, label in enumerate(era_outcomes, 1))
    + ' |',
    '|---|---|' + ':--:|' * len(era_outcomes),
]
for _, spec_label, _, _ in ERA_SPECIFICATIONS:
    for level in ERA_LEVELS:
        era_markdown_lines.append(
            f'| {spec_label} | {ERA_LABELS[level]} | ' + ' | '.join(
                era_cells[(spec_label, ERA_LABELS[level], outcome)]
                for outcome in era_outcomes) + ' |')
    era_markdown_lines.append(
        f'| | *Joint P, both eras* | ' + ' | '.join(
            f'{era_joint[(spec_label, outcome)]:.3f}'
            for outcome in era_outcomes) + ' |')
era_markdown_lines += [
    '',
    'Cells are average marginal effects in percentage points of outcome '
    'probability (95% CI), then the exposure-tolerance ratio, for each era '
    'against construction before 1992. All models adjust for log F*, log '
    'NDVI, distance into the built environment, connected neighbors, '
    'footprint area and fire, with standard errors clustered on the 250 m '
    'grid cell. Specification (1) leaves roof material out, so roof upgrades '
    'required by the codes count toward the era; (2) adds it; (3) adds a '
    'linear vintage trend, so each era term is a step at code adoption; (4) '
    'repeats (2) in Palisades, where assessed structures lie in Very High '
    'Fire Hazard Severity Zones. Era is assigned from the assessor '
    'construction year, which does not record retrofits, re-roofing or '
    'whether the code applied to a given parcel. '
    '\\*\\*\\**P* < 0.001, \\*\\**P* < 0.01, \\**P* < 0.05 '
    '(coefficient test).',
]
era_markdown = '\n'.join(era_markdown_lines)
(RESULTS / 'tableS_code_era_effects.md').write_text(
    era_markdown + '\n', encoding='utf-8')
display(era_counts.round(3))
display(Markdown(era_markdown))
print('saved', RESULTS / 'tableS_code_era_effects.csv')
print('saved', RESULTS / 'tableS_code_era_effects.md')

### Supplementary table — exposure slopes by group

*Supplement — Supplementary Table S4.*

Column (4) of the regression table reports interactions, so the exposure slope
for any particular group is a sum of coefficients rather than a single entry,
and its uncertainty is not the sum of their standard errors. Each slope below
is therefore formed as a linear combination of the fitted coefficients, with a
standard error taken from the cluster-robust covariance, and rendered as a
standalone supplementary table. Slopes are read at mean vegetation and mean
vintage and, except where distance is varied, at mean distance, which centering
makes the default. The final column converts each slope into the odds
multiplier for a 10% increase in OpenView 3D $F^*$.

In [ ]:
# Exposure slopes by group. Each is a linear combination of the model-(3)
# coefficients, so the standard error comes from the cluster-robust covariance
# rather than from adding coefficients by hand.
interaction_model = dict(
    (key, model) for key, _, _, model in FITTED_MODELS)['exposure_interactions']
interaction_parameters = list(interaction_model.params.index)
interaction_covariance = interaction_model.cov_params().to_numpy()
DEPTH_PERCENTILES = [10, 90]


def _exposure_slope(extra_terms: dict[str, float]) -> dict[str, float]:
    """Combine the exposure main effect with the named interaction terms."""
    weights = np.zeros(len(interaction_parameters))
    weights[interaction_parameters.index('log_F_star')] = 1.
    for term, value in extra_terms.items():
        weights[interaction_parameters.index(term)] = value
    slope = float(weights @ interaction_model.params.to_numpy())
    standard_error = float(np.sqrt(
        weights @ interaction_covariance @ weights))
    return {
        'exposure_slope': slope, 'standard_error': standard_error,
        'ci_lo': slope - 1.96 * standard_error,
        'ci_hi': slope + 1.96 * standard_error,
        'odds_ratio_per_10pct': float(np.exp(slope * np.log1p(.10))),
        'or_lo_per_10pct': float(np.exp(
            (slope - 1.96 * standard_error) * np.log1p(.10))),
        'or_hi_per_10pct': float(np.exp(
            (slope + 1.96 * standard_error) * np.log1p(.10))),
    }


slope_rows = [{'grouping': 'Roof material', 'group': 'Asphalt (reference)',
               **_exposure_slope({})}]
for level in ROOF_ORDER:
    term = f'log_F_star:{ROOF_TERM}[T.{level}]'
    if term in interaction_parameters:
        slope_rows.append({'grouping': 'Roof material', 'group': f'{level}',
                           **_exposure_slope({term: 1.})})
for percentile in DEPTH_PERCENTILES:
    centered = float(np.percentile(regression_sample.year_decade, percentile))
    year = centered * 10 + samples.year_reference + 10 * CENTRING['year_decade']
    slope_rows.append({
        'grouping': 'Building vintage',
        'group': f'{percentile}th percentile ({year:.0f})',
        **_exposure_slope({'log_F_star:year_decade': centered})})
for percentile in DEPTH_PERCENTILES:
    centered = float(np.percentile(regression_sample.log_n_visible, percentile))
    connected = float(np.exp(centered + CENTRING['log_n_visible']))
    slope_rows.append({
        'grouping': 'Connected neighbors',
        'group': f'{percentile}th percentile ({connected:,.0f})',
        **_exposure_slope({'log_F_star:log_n_visible': centered})})
for percentile in DEPTH_PERCENTILES:
    centered = float(np.percentile(regression_sample.log_influence_depth,
                                  percentile))
    feet = float(np.exp(centered + CENTRING['log_influence_depth']))
    slope_rows.append({
        'grouping': 'Distance into built environment',
        'group': f'{percentile}th percentile ({feet:,.0f} ft)',
        **_exposure_slope({'log_F_star:log_influence_depth': centered})})
exposure_slopes = pd.DataFrame(slope_rows)
exposure_slopes.to_csv(
    RESULTS / 'tableS_construction_exposure_slopes.csv', index=False)

SLOPE_TITLE = ('**Supplementary Table | Exposure slopes by property and site '
               'characteristic.**')
slope_display = exposure_slopes.assign(**{
    'Exposure slope (95% CI)': lambda frame: frame.apply(
        lambda row: (f'{row.exposure_slope:.3f} '
                     f'({row.ci_lo:.3f} to {row.ci_hi:.3f})'
                     ).replace('-', '−'), axis=1),
    'Odds ratio per +10% F* (95% CI)': lambda frame: frame.apply(
        lambda row: (f'{row.odds_ratio_per_10pct:.3f} '
                     f'({row.or_lo_per_10pct:.3f} to '
                     f'{row.or_hi_per_10pct:.3f})'), axis=1),
})[['grouping', 'group', 'Exposure slope (95% CI)',
    'Odds ratio per +10% F* (95% CI)']]

slope_columns = ['Group', 'Exposure slope (95% CI)',
                 'Odds ratio per +10% F* (95% CI)']
slope_lines = [SLOPE_TITLE, '', '| ' + ' | '.join(slope_columns) + ' |',
               '|---|:--:|:--:|']
for grouping, block in slope_display.groupby('grouping', sort=False):
    slope_lines.append(f'| **{grouping}** | | |')
    for _, row in block.iterrows():
        slope_lines.append(
            f'| {row.group} | {row["Exposure slope (95% CI)"]} | '
            f'{row["Odds ratio per +10% F* (95% CI)"]} |')
slope_lines += [
    '',
    'Each slope is the association between log realized coupling and the '
    'log-odds of destruction for the stated group, formed as a linear '
    'combination of the coefficients of column (4) of the regression table and '
    'given a standard error from the same cluster-robust covariance; adding '
    'the coefficients by hand would not yield a valid interval. Because '
    'continuous predictors are centered, each slope is read at mean vegetation '
    'and mean vintage, and at mean distance except in the rows where distance '
    'is varied. Percentiles are of the estimation sample. The final column '
    'converts each slope into the odds multiplier for a 10% increase in '
    '*F**. The distance rows should be read as exploratory: distance is the '
    'one covariate whose association differs between the two fires and does '
    'not survive restricting exposure to up-fire neighbors.',
]
exposure_slope_markdown = '\n'.join(slope_lines)
(RESULTS / 'tableS_construction_exposure_slopes.md').write_text(
    exposure_slope_markdown + '\n', encoding='utf-8')

display(Markdown(exposure_slope_markdown))
print('saved', RESULTS / 'tableS_construction_exposure_slopes.csv')
print('saved', RESULTS / 'tableS_construction_exposure_slopes.md')

### Predicted destruction across the exposure range

*Supplement — Extended Data Fig. 7.*

The interactions are easier to read as fitted curves than as coefficients.
Predicted probability of destruction is plotted against realized coupling over
the central 95% of the observed range, separately by roof material, building vintage, urban depth, OpenView
graph connectivity and documented defense, with the remaining
covariates held at their sample means for an asphalt-roofed structure in
Eaton. Bands are 95% intervals from the cluster-robust covariance. This is a
candidate for a main-text figure, with the regression table moving to
Supplementary Information.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from patsy import build_design_matrices

from src.viz.style import apply_style

fitted_by_key = dict((key, model) for key, _, _, model in FITTED_MODELS)
interaction_model = fitted_by_key['exposure_interactions']
defense_model = fitted_by_key['documented_defense']
exposure_grid = np.linspace(
    float(np.percentile(regression_sample.log_F_star, 2.5)),
    float(np.percentile(regression_sample.log_F_star, 97.5)), 80)
exposure_axis = np.exp(exposure_grid + CENTRING['log_F_star'])


def _predicted_probability(model, **overrides):
    """Predicted P(destroyed) with a cluster-robust 95% band."""
    scenario = pd.DataFrame({
        'log_F_star': exposure_grid,
        'roof_class': overrides.get('roof_class', 'Asphalt'),
        'log_ndvi': 0.,
        'year_decade': overrides.get('year_decade', 0.),
        'log_n_visible': overrides.get('log_n_visible', 0.),
        'log_influence_depth': overrides.get('log_influence_depth', 0.),
        'log_footprint': overrides.get('log_footprint', 0.),
        'defended_structure': overrides.get('defended_structure', 0),
        'fire': 'EATON',
    })
    design = np.asarray(build_design_matrices(
        [model.model.data.design_info], scenario)[0])
    covariance = model.cov_params().to_numpy()
    linear = design @ model.params.to_numpy()
    standard_error = np.sqrt(np.einsum(
        'ij,jk,ik->i', design, covariance, design))
    def _logistic(values):
        return 1 / (1 + np.exp(-values))
    return (_logistic(linear), _logistic(linear - 1.96 * standard_error),
            _logistic(linear + 1.96 * standard_error))


def _percentile_pair(column, percentiles=DEPTH_PERCENTILES):
    return [float(np.percentile(regression_sample[column], percentile))
            for percentile in percentiles]


depth_low, depth_high = _percentile_pair('log_influence_depth')
vintage_old, vintage_new = _percentile_pair('year_decade')
surround_low, surround_high = _percentile_pair('log_n_visible')


def _vintage_year(centered):
    return (centered + CENTRING['year_decade']) * 10 + samples.year_reference


PANELS = [
    ('Roof material', interaction_model, [
        ('Asphalt', {'roof_class': 'Asphalt'}, '#0072B2'),
        ('Tile', {'roof_class': 'Tile'}, '#D55E00'),
        ('Metal', {'roof_class': 'Metal'}, '#009E73')]),
    ('Building vintage', interaction_model, [
        (f'{_vintage_year(vintage_old):.0f} (10th pct)',
         {'year_decade': vintage_old}, '#FDBE85'),
        (f'{_vintage_year(vintage_new):.0f} (90th pct)',
         {'year_decade': vintage_new}, '#A63603')]),
    ('Distance into built environment', interaction_model, [
        (f'{np.exp(depth_low + CENTRING["log_influence_depth"]):,.0f} ft '
         '(10th pct)', {'log_influence_depth': depth_low}, '#56B4E9'),
        (f'{np.exp(depth_high + CENTRING["log_influence_depth"]):,.0f} ft '
         '(90th pct)', {'log_influence_depth': depth_high}, '#0072B2')]),
    ('Connected neighbors (pre-fire)', interaction_model, [
        (f'{np.exp(surround_low + CENTRING["log_n_visible"]):,.0f} neighbors '
         '(10th pct)', {'log_n_visible': surround_low}, '#BCBDDC'),
        (f'{np.exp(surround_high + CENTRING["log_n_visible"]):,.0f} neighbors '
         '(90th pct)', {'log_n_visible': surround_high}, '#54278F')]),
    ('Documented defense', defense_model, [
        ('Undefended', {'defended_structure': 0}, '#D55E00'),
        ('Documented defense', {'defended_structure': 1}, '#0072B2')]),
]

apply_style()
fig, axes = plt.subplots(2, 3, figsize=(8.1, 4.9), sharex=True,
                         sharey=True)
prediction_rows = []
for panel_index, (panel_title, model, series) in enumerate(PANELS):
    ax = axes.flat[panel_index]
    for series_label, overrides, color in series:
        probability, lower, upper = _predicted_probability(model, **overrides)
        ax.fill_between(exposure_axis, lower, upper, color=color, alpha=.16,
                        linewidth=0)
        ax.plot(exposure_axis, probability, color=color, lw=2.0,
                label=series_label)
        prediction_rows.append(pd.DataFrame({
            'panel': panel_title, 'series': series_label,
            'F_star': exposure_axis, 'probability': probability,
            'ci_lo': lower, 'ci_hi': upper}))
    ax.set_xscale('log')
    ax.set_ylim(0, 1)
    ax.grid(axis='y', color='#E2E2E2', lw=.5)
    ax.grid(axis='x', color='#EFEFEF', lw=.4)
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(labelsize=6.6)
    ax.text(0, 1.03, f'{"abcdef"[panel_index]}   {panel_title}',
            transform=ax.transAxes, fontsize=7.8, fontweight='bold',
            va='bottom')
    ax.legend(loc='upper left', fontsize=6.1, frameon=False,
              handlelength=1.3, borderpad=.1, labelspacing=.22)
axes.flat[len(PANELS)].set_visible(False)
for ax in list(axes[1, :2]) + [axes[0, 2]]:
    ax.set_xlabel('Realized coupling, $F^*$', fontsize=7.4)
    ax.tick_params(labelbottom=True)
for ax in axes[:, 0]:
    ax.set_ylabel('Predicted P(destroyed)', fontsize=7.4)
    ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
fig.subplots_adjust(left=.075, right=.99, top=.93, bottom=.10,
                    wspace=.10, hspace=.30)

exposure_predictions = pd.concat(prediction_rows, ignore_index=True)
exposure_predictions.to_csv(
    RESULTS / 'ED_fig_exposure_interactions_source.csv', index=False)
exposure_stem = save_figure(
    fig, 'construction_exposure_interactions', 'supplementary',
    source=exposure_predictions, dpi=600,
)
plt.show()
print('Panels a–d are the interaction model on exposed, undefended structures; '
      'panel e is the additive defense model, which retains defended '
      'structures. Curves hold the remaining covariates at their sample means '
      'for an asphalt roof in Eaton; bands are 95% cluster-robust intervals.')
print('saved', exposure_stem.with_suffix('.png'))

### Robustness: consistency between fires, and reported tolerance ratios

*Supplement — Supplementary Table S3.*

The first check asks whether the reported associations depend on pooling the
two fires: every attribute, the exposure slope and distance are interacted
with fire and the interactions are tested jointly on the cluster-robust
covariance, so a term is reported as consistent between fires when its
interaction is not distinguishable from zero. The second table converts the
reporting model into exposure-tolerance ratios on interpretable spans, an
interquartile range for the logged predictors rather than a log unit, which is
the form the results section quotes.

In [ ]:
# Do the associations differ between the two fires? Every attribute, the
# exposure slope and distance are interacted with fire and tested jointly.
VARYING_TERMS = (f'log_F_star + {ROOF_TERM} + log_ndvi + year_decade '
                 f'+ {SITE_TERM} + {DEPTH_TERM}')
fire_heterogeneity_model = fit_clustered_logit(
    regression_sample,
    f'is_destroyed ~ {BASE_TERMS} + {DEPTH_TERM} + C(fire):({VARYING_TERMS})')
fire_interaction_terms = [term for term in fire_heterogeneity_model.params.index
                          if 'C(fire)' in term and ':' in term]
restriction = np.zeros((len(fire_interaction_terms),
                        len(fire_heterogeneity_model.params)))
for row, term in enumerate(fire_interaction_terms):
    restriction[row, list(
        fire_heterogeneity_model.params.index).index(term)] = 1
fire_joint_test = fire_heterogeneity_model.wald_test(restriction, scalar=True)
fire_heterogeneity = pd.DataFrame([{
    'term': term, 'term_label': _term_label(term.split(':', 1)[1]),
    'coefficient': float(fire_heterogeneity_model.params[term]),
    'cluster_robust_se': float(fire_heterogeneity_model.bse[term]),
    'p_value': float(fire_heterogeneity_model.pvalues[term]),
} for term in fire_interaction_terms])
fire_heterogeneity['consistent_between_fires'] = (
    fire_heterogeneity.p_value.ge(.05))

fire_heterogeneity.to_csv(
    RESULTS / 'tableS_construction_fire_heterogeneity.csv', index=False)

print(f'Fire interactions tested jointly: '
      f'chi2({len(fire_interaction_terms)}) = '
      f'{float(fire_joint_test.statistic):.1f}, '
      f'P = {float(fire_joint_test.pvalue):.4g}')
display(fire_heterogeneity.round(4))
print('saved', RESULTS / 'tableS_construction_fire_heterogeneity.csv')

### Distance into the urban core without the exposure control

Realized coupling lies on the path from site position to destruction: structures deeper into the built environment have fewer destroyed neighbours in view, so model 1 estimates the part of the depth association that does *not* run through exposure. Refitting the same specification without log $F^*$ gives the total association instead. The two answers point in opposite directions, so the comparison is reported as odds ratios, which are defined in both fits, rather than as exposure-tolerance ratios, which need a log $F^*$ coefficient.

In [ ]:
# Distance into the urban core with the exposure control removed. F* lies on
# the path from site position to destruction: deeper structures have fewer
# destroyed neighbours in view, so conditioning on F* estimates the part of
# the depth association that does not run through realized exposure. Dropping
# it gives the total association. Without a log F* coefficient there is no
# exposure-tolerance ratio, so both fits are read as odds ratios.
fitted_by_key = dict((key, model) for key, _, _, model in FITTED_MODELS)
depth_without_exposure = fitted_by_key['no_exposure']
depth_with_exposure = fitted_by_key['site_depth']

EQUIVALENT_STEP = np.log1p(0.10)  # a 10% increase in the logged predictor
depth_rows = []
for specification, model in (
    ('Model 1, adjusted for log F*', depth_with_exposure),
    ('Model 0, without log F*', depth_without_exposure),
):
    for term, term_label in (
        (DEPTH_TERM, 'Distance into urban core'),
        (SITE_TERM, 'Connected neighbors'),
    ):
        low, high = np.percentile(regression_sample[term], [25, 75])
        span = float(high - low)
        quartiles = np.exp(np.array([low, high]) + CENTRING[term])
        coefficient = float(model.params[term])
        standard_error = float(model.bse[term])
        depth_rows.append({
            'specification': specification,
            'term': term_label,
            'coefficient': coefficient,
            'cluster_robust_se': standard_error,
            'p_value': float(model.pvalues[term]),
            'q1': float(quartiles[0]),
            'q3': float(quartiles[1]),
            'odds_ratio_iqr': float(np.exp(coefficient * span)),
            'ci_lo_iqr': float(np.exp((coefficient - 1.96 * standard_error) * span)),
            'ci_hi_iqr': float(np.exp((coefficient + 1.96 * standard_error) * span)),
            'odds_ratio_per_10pct': float(np.exp(coefficient * EQUIVALENT_STEP)),
        })
depth_exposure_comparison = pd.DataFrame(depth_rows)
depth_exposure_comparison.to_csv(
    RESULTS / 'depth_without_exposure_comparison.csv', index=False,
)

coefficients = (
    depth_exposure_comparison.set_index(['term', 'specification']).coefficient
)
for term_label in ('Distance into urban core', 'Connected neighbors'):
    total = coefficients[(term_label, 'Model 0, without log F*')]
    direct = coefficients[(term_label, 'Model 1, adjusted for log F*')]
    turned = 'sign reverses' if total * direct < 0 else 'same sign'
    print(f'{term_label}: {total:+.3f} unadjusted -> {direct:+.3f} adjusted '
          f'for log F* ({turned})')

# Both terms predict exposure itself, which is why conditioning on F* moves
# them so far: the unadjusted fit carries the exposure they generate.
for term, term_label in ((DEPTH_TERM, 'Distance into urban core'),
                         (SITE_TERM, 'Connected neighbors')):
    exposure_fit = fit_clustered_logit.__globals__['smf'].ols(
        f'log_F_star ~ {term}', data=regression_sample,
    ).fit(cov_type='cluster',
          cov_kwds={'groups': regression_sample.grid_id})
    print(f'  log F* on {term_label}: {float(exposure_fit.params[term]):+.3f} '
          f'per log unit (R2 = {exposure_fit.rsquared:.3f})')
print(
    f"McFadden pseudo-R2: {1 - depth_with_exposure.llf / depth_with_exposure.llnull:.3f} "
    f"with log F*, {1 - depth_without_exposure.llf / depth_without_exposure.llnull:.3f} without"
)
pd.set_option('display.width', 200)
display(depth_exposure_comparison.round(
    {'coefficient': 3, 'cluster_robust_se': 3, 'p_value': 4, 'q1': 0, 'q3': 0,
     'odds_ratio_iqr': 3, 'ci_lo_iqr': 3, 'ci_hi_iqr': 3,
     'odds_ratio_per_10pct': 4}
))

### Distance from the wildland across exposure strata

*Supplement — Supplementary Fig. S?.*

At equal realized coupling, the distance association changes sign with
exposure (column (4), log $F^*$ × distance). This figure shows the same pattern
without a model: the share of structures destroyed against distance to the
Influence Zone boundary, within strata of $F^*$. $F^*$ counts exposure from
destroyed structures only, so in the lowest strata a structure's exposure
comes mainly from the wildland, and destruction should fall with distance if
ember cast declines away from the boundary; in the highest strata exposure is
dominated by burning neighbors. Points are binned shares with 95% intervals
clustered on the 250 m grid cell and lines are LOWESS fits, as in Fig. 1c,d.
Strata are the $F^*$ quartiles of the regression sample, plus structures with
no destroyed neighbor in view ($F^* = 0$), which lie outside the regression
sample but meet its undefended, complete-data and site criteria.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex, to_rgb
from matplotlib.lines import Line2D
from matplotlib.ticker import FixedFormatter, FixedLocator, PercentFormatter
from statsmodels.nonparametric.smoothers_lowess import lowess

from src.viz.style import FIRE_COLORS, apply_style

# Destruction against a site characteristic, within strata of realized
# coupling. F* counts exposure from destroyed structures only, so where it is
# low a structure's exposure comes mainly from the wildland. The F* = 0
# stratum lies outside the regression sample (which requires F* > 0) but meets
# the same undefended, complete-data and site criteria. This cell draws
# distance to the Influence boundary; the next reuses the helpers for density.
PROFILE_BINS = 10
MIN_BIN_STRUCTURES = 30

unexposed = analysis[
    analysis.exposed.ne(1) & ~analysis.defended.astype(bool)
    & analysis.roof_class.astype(str).ne('Unknown')
    & analysis.year_built.notna()]
unexposed_sample = _prepare_sample(unexposed)

exposure_edges = np.percentile(
    np.exp(regression_sample.log_F_star + CENTRING['log_F_star']), [25, 50, 75])
EXPOSURE_STRATA = [
    'No destroyed neighbor in view (F* = 0)',
    f'F* < {exposure_edges[0]:.3f} (lowest quartile)',
    f'F* {exposure_edges[0]:.3f}–{exposure_edges[1]:.3f}',
    f'F* {exposure_edges[1]:.3f}–{exposure_edges[2]:.2f}',
    f'F* > {exposure_edges[2]:.2f} (highest quartile)',
]
exposed_profile_input = regression_sample.assign(
    stratum=1 + np.digitize(
        np.exp(regression_sample.log_F_star + CENTRING['log_F_star']),
        exposure_edges))
exposure_strata = pd.concat([
    unexposed_sample.assign(stratum=0), exposed_profile_input,
], ignore_index=True)[['fire', 'grid_id', 'is_destroyed', 'stratum',
                       'influence_depth_ft', 'n_visible']]
exposure_strata['is_destroyed'] = exposure_strata.is_destroyed.astype(int)


def _stratum_profile(frame: pd.DataFrame, column: str) -> pd.DataFrame:
    """Binned destruction shares by fire and stratum, as in Fig. 1c,d.

    Bins are deciles of the log characteristic within each fire and stratum,
    so every stratum is resolved across its own range; standard errors are
    clustered on the 250 m grid cell.
    """
    rows = []
    for (fire_name, stratum), group in frame.groupby(['fire', 'stratum']):
        group = group.assign(ln_value=np.log(group[column]))
        group = group.assign(value_bin=pd.qcut(
            group.ln_value, PROFILE_BINS, labels=False, duplicates='drop'))
        for value_bin, cell in group.groupby('value_bin'):
            if len(cell) < MIN_BIN_STRUCTURES:
                continue
            y = cell.is_destroyed.to_numpy(float)
            share = float(y.mean())
            scores = (pd.Series(y - share, index=cell.index)
                      .groupby(cell.grid_id.astype(str)).sum().to_numpy())
            n_clusters = len(scores)
            standard_error = (np.sqrt(n_clusters / (n_clusters - 1)
                                      * np.sum(scores ** 2) / len(cell) ** 2)
                              if n_clusters > 1 else np.nan)
            rows.append({
                'variable': column, 'fire': fire_name,
                'stratum': int(stratum),
                'stratum_label': EXPOSURE_STRATA[stratum],
                'value_bin': int(value_bin), 'structures': len(cell),
                'clusters_250m': n_clusters,
                'value': float(np.exp(cell.ln_value.mean())),
                'ln_value': float(cell.ln_value.mean()),
                'destroyed_share': share,
                'share_lo': max(0., share - 1.96 * standard_error),
                'share_hi': min(1., share + 1.96 * standard_error),
            })
    return pd.DataFrame(rows)


def _stratum_bands(frame: pd.DataFrame, column: str, edges: list[float],
                   labels: list[str]) -> pd.DataFrame:
    """Destruction shares over fixed bands of the characteristic."""
    bands = (frame.assign(band=pd.cut(frame[column], edges, labels=labels))
             .groupby(['fire', 'stratum', 'band'], observed=True)
             .is_destroyed.agg(structures='size', destroyed_share='mean')
             .reset_index())
    bands['stratum_label'] = bands.stratum.map(dict(enumerate(EXPOSURE_STRATA)))
    return bands


def _band_table(bands: pd.DataFrame) -> pd.DataFrame:
    """Bands with fewer structures than a plotted bin are too thin to quote."""
    return (bands[bands.structures.ge(MIN_BIN_STRUCTURES)]
            .pivot_table(index=['fire', 'stratum_label'], columns='band',
                         values='destroyed_share', observed=True, sort=False)
            .mul(100).round(1))


def _stratum_ramp(fire_name: str) -> list[str]:
    """Light-to-dark shades of the fire color for the four F* > 0 strata."""
    base = np.array(to_rgb(FIRE_COLORS[fire_name]))
    white, black = np.ones(3), np.zeros(3)
    return [to_hex(white + weight * (base - white)) for weight in (.3, .58, .85)] \
        + [to_hex(base + .3 * (black - base))]


UNEXPOSED_COLOR = '#8F8F8F'
LEGEND_RAMP = ['#C4C4C4', '#8C8C8C', '#5A5A5A', '#262626']


def _plot_stratum_profile(profile: pd.DataFrame, *, xlabel: str,
                          ticks: list[float], xlim: tuple[float, float],
                          title: str):
    """Two fire panels of destruction share against the characteristic."""
    apply_style()
    fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.6), sharey=True)
    for col_i, fire_name in enumerate(['EATON', 'PALISADES']):
        ax = axes[col_i]
        ramp = _stratum_ramp(fire_name)
        for stratum in range(len(EXPOSURE_STRATA)):
            series = profile[profile.fire.eq(fire_name)
                             & profile.stratum.eq(stratum)].sort_values('value')
            if series.empty:
                continue
            unexposed_series = stratum == 0
            color = UNEXPOSED_COLOR if unexposed_series else ramp[stratum - 1]
            if len(series) >= 4:
                smooth = lowess(series.destroyed_share, series.ln_value,
                                frac=.6, return_sorted=True)
                ax.plot(np.exp(smooth[:, 0]), np.clip(smooth[:, 1], 0, 1),
                        color=color, lw=1.15 if unexposed_series else 1.65,
                        ls=':' if unexposed_series else '-', zorder=3)
            yerr = np.vstack([series.destroyed_share - series.share_lo,
                              series.share_hi - series.destroyed_share])
            ax.errorbar(series.value, series.destroyed_share,
                        yerr=np.clip(yerr, 0, None), fmt='o', ms=3.0, mew=.7,
                        color=color, ecolor=color, elinewidth=.45, alpha=.85,
                        capsize=1.2, lw=0,
                        mfc='white' if unexposed_series else color, zorder=4)
        ax.text(0, 1.035, f'{"ab"[col_i]}   {fire_name.title()}',
                transform=ax.transAxes, ha='left', va='bottom', fontsize=8.8,
                fontweight='bold', clip_on=False)
        ax.set_xscale('log'); ax.set_ylim(0, 1); ax.set_xlim(*xlim)
        ax.set_xticks(ticks)
        ax.xaxis.set_major_formatter(FixedFormatter(
            [f'{value:,}' for value in ticks]))
        ax.xaxis.set_minor_locator(FixedLocator([]))
        ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
        ax.set_xlabel(xlabel, labelpad=4, fontsize=7.1)
        ax.grid(axis='y', color='#D7D7D7', lw=.5, alpha=.75)
        ax.grid(axis='x', color='#E5E5E5', lw=.4, alpha=.65)
        ax.spines[['top', 'right']].set_visible(False)
        ax.tick_params(labelsize=6.8)
    axes[0].set_ylabel('Share of structures destroyed', labelpad=5,
                       fontsize=7.4)
    legend_handles = [Line2D([0], [0], color=UNEXPOSED_COLOR, lw=1.15, ls=':',
                             marker='o', markersize=3.6, mfc='white',
                             label=EXPOSURE_STRATA[0])]
    legend_handles += [Line2D([0], [0], color=shade, lw=1.5, marker='o',
                              markersize=3.6, label=label)
                       for shade, label in zip(LEGEND_RAMP, EXPOSURE_STRATA[1:])]
    fig.legend(handles=legend_handles, frameon=False, loc='lower center',
               bbox_to_anchor=(.5, .005), ncol=3, fontsize=6.6)
    fig.suptitle(title, x=.075, ha='left', y=.99, fontsize=10.2,
                 fontweight='bold')
    fig.subplots_adjust(left=.085, right=.99, top=.86, bottom=.245, wspace=.08)
    return fig


depth_strata_profile = _stratum_profile(exposure_strata, 'influence_depth_ft')
depth_strata_profile.to_csv(
    RESULTS / 'SI_fig_depth_by_exposure_stratum_source.csv', index=False)
# The same shares over the depth bands of Fig. 1, for quoting in the text.
depth_strata_bands = _stratum_bands(
    exposure_strata, 'influence_depth_ft', [0, 1000, 2500, 5000, 7500, np.inf],
    ['0–1,000', '1,000–2,500', '2,500–5,000', '5,000–7,500', '>7,500'])
depth_strata_bands.to_csv(
    RESULTS / 'SI_depth_by_exposure_stratum_bands.csv', index=False)

fig = _plot_stratum_profile(
    depth_strata_profile, xlabel='Distance to Influence boundary (ft)',
    ticks=[500, 1000, 2500, 5000, 10000], xlim=(300, 15000),
    title='Distance from the wildland by realized coupling')
depth_strata_stem = save_figure(
    fig, 'SI_depth_by_exposure_stratum', 'supplementary',
    source=depth_strata_profile)
plt.show()
display(_band_table(depth_strata_bands))
print('saved', depth_strata_stem.with_suffix('.png'))

### Density across exposure strata

*Supplement — Supplementary Fig. S?.*

At equal realized coupling, more connected neighbors is associated with lower
odds of destruction (column (1)), and that association also changes sign with
exposure (column (4), log $F^*$ × connected neighbors). This figure repeats
the distance construction above with the pre-fire count of connected neighbors
on the horizontal axis. If sparse structures at low $F^*$ are exposed to the
wildland in ways $F^*$ does not measure, destruction should fall with neighbor
count in the lowest strata and not in the highest.

In [ ]:
# Connected neighbors within the same exposure strata, reusing the helpers
# above. The count is fixed pre-fire geometry: every structure with a visible
# view of the focal building, burned or not.
density_strata_profile = _stratum_profile(exposure_strata, 'n_visible')
density_strata_profile.to_csv(
    RESULTS / 'SI_fig_density_by_exposure_stratum_source.csv', index=False)
density_strata_bands = _stratum_bands(
    exposure_strata, 'n_visible', [0, 10, 20, 30, 50, np.inf],
    ['1–10', '11–20', '21–30', '31–50', '>50'])
density_strata_bands.to_csv(
    RESULTS / 'SI_density_by_exposure_stratum_bands.csv', index=False)

fig = _plot_stratum_profile(
    density_strata_profile, xlabel='Connected neighbors (pre-fire count)',
    ticks=[2, 5, 10, 20, 50, 100], xlim=(1.5, 120),
    title='Density by realized coupling')
density_strata_stem = save_figure(
    fig, 'SI_density_by_exposure_stratum', 'supplementary',
    source=density_strata_profile)
plt.show()
display(_band_table(density_strata_bands))
print('saved', density_strata_stem.with_suffix('.png'))

### Standardized protective margins

*Supplement — Extended Data Fig. 6.*

ED03 reports the reduction in destruction probability associated with each
attribute across realized coupling. That margin is a contrast *within one
outcome*: at each value of OpenView 3D $F^*$ the whole population is predicted twice, once
with the vulnerable attribute setting and once with the protective one, and the
two standardized probabilities are differenced. This cell repeats that
construction on the interaction model fitted here, which adds distance into the
built environment as a fourth contrast, with bands from draws of the
cluster-robust coefficient covariance.

In [ ]:
from matplotlib.ticker import LogFormatterMathtext
from scipy.special import expit

from src.viz.style import log_grid

MARGIN_GRID = 40
MARGIN_DRAWS = 200
MARGIN_SEED = 20250808

margin_grid = np.geomspace(
    float(np.exp(np.percentile(regression_sample.log_F_star, 2)
                 + CENTRING['log_F_star'])),
    float(np.exp(np.percentile(regression_sample.log_F_star, 98)
                 + CENTRING['log_F_star'])), MARGIN_GRID)
observed_F_star = np.exp(regression_sample.log_F_star
                         + CENTRING['log_F_star']).to_numpy()


def _standardized_probability(model, frame, overrides):
    """Average predicted probability over the population under one setting."""
    design = np.asarray(build_design_matrices(
        [model.model.data.design_info], frame.assign(**overrides),
        return_type='dataframe')[0])
    return design, expit(design @ np.asarray(model.params))


def _margin_curve(model, vulnerable, protective, draws):
    """Vulnerable-minus-protective margin in percentage points, with a band."""
    point, lower, upper = [], [], []
    for exposure in margin_grid:
        common = {'log_F_star': np.log(exposure) - CENTRING['log_F_star']}
        design_v, p_v = _standardized_probability(
            model, regression_sample, {**common, **vulnerable})
        design_p, p_p = _standardized_probability(
            model, regression_sample, {**common, **protective})
        point.append(100 * float(np.mean(p_v - p_p)))
        simulated = []
        for start in range(0, len(draws), 25):
            block = draws[start:start + 25]
            simulated.extend(100 * (expit(design_v @ block.T).mean(axis=0)
                                    - expit(design_p @ block.T).mean(axis=0)))
        low, high = np.percentile(simulated, [2.5, 97.5])
        lower.append(float(low)); upper.append(float(high))
    return np.array(point), np.array(lower), np.array(upper)


interaction_model = dict(
    (key, model) for key, _, _, model in FITTED_MODELS)['exposure_interactions']

rng = np.random.default_rng(MARGIN_SEED)
interaction_draws = rng.multivariate_normal(
    np.asarray(interaction_model.params),
    np.asarray(interaction_model.cov_params()), size=MARGIN_DRAWS,
    check_valid='ignore')

ndvi_low, ndvi_high = np.percentile(regression_sample.log_ndvi, [25, 75])
year_old, year_new = np.percentile(regression_sample.year_decade, [25, 75])
depth_near, depth_far = np.percentile(
    regression_sample.log_influence_depth, [25, 75])
surround_low, surround_high = np.percentile(
    regression_sample.log_n_visible, [25, 75])
MARGIN_CONTRASTS = {
    'Tile roofing': ({'roof_class': 'Asphalt'}, {'roof_class': 'Tile'},
                     'Tile rather than asphalt', '#2E5B82'),
    'Lower vegetation': ({'log_ndvi': ndvi_high}, {'log_ndvi': ndvi_low},
                         '25th rather than 75th percentile NDVI', '#3D6B45'),
    'Newer construction': ({'year_decade': year_old},
                           {'year_decade': year_new},
                           '75th rather than 25th percentile year', '#A66F00'),
    'Greater distance': ({'log_influence_depth': depth_near},
                         {'log_influence_depth': depth_far},
                         '75th rather than 25th percentile distance',
                         '#7B4173'),
    'More connected neighbors': ({'log_n_visible': surround_low},
                                  {'log_n_visible': surround_high},
                                  '75th rather than 25th percentile connected '
                                  'neighbors', '#2A6F72'),
}

margin_rows, margin_summary_rows = [], []
for attribute, (vulnerable, protective, contrast, _) in MARGIN_CONTRASTS.items():
    point, lower, upper = _margin_curve(
        interaction_model, vulnerable, protective, interaction_draws)
    peak = int(np.nanargmax(point))
    for index, exposure in enumerate(margin_grid):
        margin_rows.append({
            'attribute': attribute, 'contrast': contrast,
            'F_star': float(exposure), 'margin_pp': float(point[index]),
            'ci_lo': float(lower[index]), 'ci_hi': float(upper[index]),
            'exposure_percentile': float(
                100 * (observed_F_star <= exposure).mean())})
    margin_summary_rows.append({
        'attribute': attribute, 'contrast': contrast,
        'peak_margin_pp': float(point[peak]),
        'peak_F_star': float(margin_grid[peak]),
        'peak_exposure_percentile': float(
            100 * (observed_F_star <= margin_grid[peak]).mean()),
        'margin_at_p98_pp': float(point[-1])})
construction_margin_curves = pd.DataFrame(margin_rows)
construction_margin_summary = pd.DataFrame(margin_summary_rows)

construction_margin_curves.to_csv(
    RESULTS / 'ED_fig_construction_margins_source.csv', index=False)
construction_margin_summary.to_csv(
    RESULTS / 'tableS_construction_margin_summary.csv', index=False)

apply_style()
fig, ax = plt.subplots(figsize=(7.2, 4.15))
for attribute, (_, _, _, color) in MARGIN_CONTRASTS.items():
    block = (construction_margin_curves[
        construction_margin_curves.attribute.eq(attribute)]
        .sort_values('F_star'))
    ax.fill_between(block.F_star, block.ci_lo, block.ci_hi, color=color,
                    alpha=.13, linewidth=0)
    ax.plot(block.F_star, block.margin_pp, color=color, lw=2.0,
            label=attribute)
    peak = construction_margin_summary[
        construction_margin_summary.attribute.eq(attribute)].iloc[0]
    ax.plot(peak.peak_F_star, peak.peak_margin_pp, 'o', ms=5, color=color,
            mfc='white', mew=1.1, zorder=5)
    ax.annotate(f'{peak.peak_margin_pp:.1f} pp',
                (peak.peak_F_star, peak.peak_margin_pp), xytext=(4, 5),
                textcoords='offset points', fontsize=7, color=color)
ax.axhline(0, color='#4A4A4A', lw=.7)
ax.set_xscale('log')
ax.xaxis.set_major_formatter(LogFormatterMathtext())
ax.set_xlabel(r'Realized geometric coupling, $F^*$')
ax.set_ylabel('Predicted reduction in destruction\n(percentage points)')
ax.set_title('Relative advantage across exposure', loc='left', fontsize=10,
             fontweight='bold')
ax.text(.995, .02,
        'Lines: standardized predictions; shading: 95% cluster-robust '
        'confidence bands', transform=ax.transAxes, ha='right', va='bottom',
        fontsize=7, color='#555555')
log_grid(ax, axis='x')
ax.grid(True, axis='y', color='#E4E4E4', lw=.45)
ax.legend(frameon=False, loc='upper left', ncol=3, handlelength=1.8,
          columnspacing=1.4)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
fig.subplots_adjust(left=.105, right=.99, top=.88, bottom=.16)
margin_stem = save_figure(
    fig, 'construction_relative_advantage_across_exposure', 'supplementary',
    source=construction_margin_curves,
)
plt.show()

margin_caption = (
    '**Extended Data Figure | Standardized protective margins across realized '
    'coupling.** Predicted reduction in the probability of destruction '
    'associated with each contrast. At every value of $F^*$ the whole analysis '
    'population is predicted twice, once under the vulnerable setting and once '
    'under the protective setting, and the two standardized probabilities are '
    'differenced; the curves are therefore population-averaged contrasts '
    'within a single outcome, not differences between outcomes. Contrasts are '
    'tile rather than asphalt roofing, the 25th rather than the 75th '
    'percentile of near-structure NDVI, the 75th rather than the 25th '
    'percentile of construction year, and the 75th rather than the 25th '
    'percentile of distance into the built environment. Estimates come from '
    f'the exposure-interaction model fitted to {len(regression_sample):,} '
    'exposed, undefended structures; shading gives 95% intervals from '
    f'{MARGIN_DRAWS} draws of the cluster-robust coefficient covariance. Open '
    'circles mark each maximum. The connected-neighbor contrast differs in '
    'kind from the other four: the count is realized during the fire, like '
    '*F*\\* itself, rather than a pre-fire characteristic that could be '
    'altered by siting or construction. Margins are near zero at low '
    'coupling, peak '
    'within the fragility transition and decline as destruction saturates; '
    'the distance contrast alone reverses sign, and distance is also the one '
    'covariate whose association differs between the two fires and does not '
    'survive restricting exposure to up-fire neighbors, so it should be read '
    'as exploratory.')
(RESULTS / 'ED_fig_construction_margins_caption.md').write_text(
    margin_caption + '\n', encoding='utf-8')

display(Markdown(margin_caption))
display(construction_margin_summary.round(3))
print('saved', margin_stem.with_suffix('.png'))

## Cross-fire sample and composition standardization

*Separate analysis — cross-fire composition standardization.*

The sample ladder separates the change associated with excluding defense from subsequent complete-data selection. On the complete sample, each fire's five-parameter fragility curve is refitted after reweighting its roof, age-cohort and vegetation distribution toward the other fire. The symmetric composition factor is the geometric mean of the two directional swaps. These are standardized comparisons, not causal components; factor-specific shares overlap and are not additive. Confidence intervals use 200 fire-stratified case-bootstrap refits, matching the prespecified analysis.

In [ ]:
sample_ladder, standardization_table, standardization_draws = (
    cross_fire_standardization(
        samples, n_boot=N_BOOT, seed=BOOTSTRAP_SEED
    )
)
sample_ladder.to_csv(RESULTS / 'cross_fire_sample_ladder.csv', index=False)
standardization_table.to_csv(
    RESULTS / 'cross_fire_standardization.csv', index=False
)
standardization_draws.to_csv(
    RESULTS / 'cross_fire_standardization_bootstrap.csv', index=False
)
display(sample_ladder.round(5))
display(standardization_table.round(4))

## Miscellaneous tests

*Miscellaneous — checks that support statements in the text; not reported as a
numbered table.*

### How much of each association runs through realized coupling (KHB)

Columns (0) and (1) of the regression table differ only by log $F^*$, but their
coefficients cannot be differenced to find the part of an association that runs
through exposure: adding log $F^*$ changes the latent residual variance of the
logit, so the two columns are on different scales. The KHB method (Karlson,
Holm and Breen 2012) removes that rescaling. Log $F^*$ is regressed linearly on
every other regressor, and the reduced model is refitted with the residual in
its place. The residual carries the same information as log $F^*$, so the
refitted model reproduces column (1) exactly; it is uncorrelated with the
characteristics, so their coefficients recover the total association on the
column (1) scale. The total then splits exactly into the column (1)
coefficient (direct) and the log $F^*$ coefficient times the characteristic's
slope on log $F^*$ (via $F^*$). The motivating case is connected neighbors,
whose sign reverses between columns (0) and (1). The decomposition is
descriptive: $F^*$ is built from the outcomes of neighboring structures, so
conditioning on it can induce associations of its own, and the direct
component is not a causal direct effect.

In [ ]:
import statsmodels.formula.api as smf
from scipy.stats import norm

# KHB decomposition (Karlson, Holm and Breen 2012). Differencing columns (0)
# and (1) confounds the part of an association that runs through F* with
# rescaling: adding log F* shrinks the latent residual variance, so the two
# logits are on different scales. KHB instead replaces log F* in the reduced
# model with its residual from a linear regression on every other regressor.
# The residual carries the same information as log F*, so the reduced model
# keeps the full model's fit and scale, but it is uncorrelated with the
# characteristics, so their coefficients keep their total association.
KHB_KEYS = [
    (TILE_TERM, 'Tile roof, versus asphalt', 1.),
    ('log_ndvi', 'Near-structure NDVI', None),
    ('year_decade', 'Building vintage, per decade newer', 1.),
    ('log_influence_depth', 'Distance into built environment', None),
    ('log_n_visible', 'Connected neighbors (pre-fire)', None),
    ('log_footprint', 'Footprint area', None),
]
fitted_by_key = dict((key, model) for key, _, _, model in FITTED_MODELS)
khb_full = fitted_by_key['site_depth']
khb_naive = fitted_by_key['no_exposure']
khb_sample = (MODEL_SAMPLES['site_depth'][0]
              .loc[khb_full.model.data.row_labels].copy())
CHARACTERISTIC_TERMS = f'{NO_EXPOSURE_TERMS} + {DEPTH_TERM}'
exposure_on_characteristics = smf.ols(
    f'log_F_star ~ {CHARACTERISTIC_TERMS}', data=khb_sample,
).fit(cov_type='cluster', cov_kwds={'groups': khb_sample.grid_id})
khb_sample['log_F_star_residual'] = exposure_on_characteristics.resid
khb_reduced = fit_clustered_logit(
    khb_sample,
    f'is_destroyed ~ {CHARACTERISTIC_TERMS} + log_F_star_residual')

# The residualized model is a reparametrization of column (1), so its fit is
# identical and each total equals direct + (log F* coefficient) x (slope of
# log F* on the characteristic). Both identities are checked before use.
exposure_coefficient = float(khb_full.params['log_F_star'])
if not np.isclose(khb_reduced.llf, khb_full.llf, rtol=1e-8):
    raise ValueError('KHB reduced model does not reproduce the full fit.')

khb_rows = []
for term, label, span in KHB_KEYS:
    if span is None:  # logged predictors are read over their interquartile range
        low, high = np.percentile(regression_sample[term], [25, 75])
        span = float(high - low)
    slope_on_exposure = float(exposure_on_characteristics.params[term])
    direct = float(khb_full.params[term])
    total = float(khb_reduced.params[term])
    if not np.isclose(total, direct + exposure_coefficient * slope_on_exposure,
                      atol=1e-6):
        raise ValueError(f'KHB identity fails for {term}.')
    indirect = total - direct
    # Delta-method (Sobel) standard error, treating the two estimates as
    # independent, as in Karlson, Holm and Breen.
    indirect_se = float(np.sqrt(
        slope_on_exposure ** 2 * khb_full.bse['log_F_star'] ** 2
        + exposure_coefficient ** 2
        * exposure_on_characteristics.bse[term] ** 2))
    khb_rows.append({
        'term': term, 'term_label': label, 'reported_span': span,
        'without_exposure': float(khb_naive.params[term]),
        'without_exposure_se': float(khb_naive.bse[term]),
        'without_exposure_p': float(khb_naive.pvalues[term]),
        'total': total, 'total_se': float(khb_reduced.bse[term]),
        'total_p': float(khb_reduced.pvalues[term]),
        'direct': direct, 'direct_se': float(khb_full.bse[term]),
        'direct_p': float(khb_full.pvalues[term]),
        'via_exposure': indirect, 'via_exposure_se': indirect_se,
        'via_exposure_p': float(2 * norm.sf(abs(indirect / indirect_se))),
        'share_via_exposure': indirect / total,
        'rescaling_ratio': total / float(khb_naive.params[term]),
        'total_odds_ratio_span': float(np.exp(total * span)),
        'direct_odds_ratio_span': float(np.exp(direct * span)),
        'via_exposure_odds_ratio_span': float(np.exp(indirect * span)),
    })
khb_decomposition = pd.DataFrame(khb_rows)
khb_decomposition.to_csv(
    RESULTS / 'misc_khb_exposure_decomposition.csv', index=False)


def _khb_cell(estimate: float, standard_error: float, p_value: float) -> str:
    return (f'{estimate:.3f}{_stars(p_value)} '
            f'({standard_error:.3f})').replace('-', '−')


khb_markdown_lines = [
    '**KHB decomposition of each association with destruction into the part '
    'running through realized coupling.**',
    '',
    '| Characteristic | Without F* (column 0) | Total (KHB) | Direct '
    '(column 1) | Via F* | Share via F* | Odds ratio, total / direct / via F* '
    '|',
    '|---|:--:|:--:|:--:|:--:|:--:|:--:|',
    *[f'| {row.term_label} '
      f'| {_khb_cell(row.without_exposure, row.without_exposure_se, row.without_exposure_p)} '
      f'| {_khb_cell(row.total, row.total_se, row.total_p)} '
      f'| {_khb_cell(row.direct, row.direct_se, row.direct_p)} '
      f'| {_khb_cell(row.via_exposure, row.via_exposure_se, row.via_exposure_p)} '
      f'| {100 * row.share_via_exposure:.0f}% '
      f'| {row.total_odds_ratio_span:.2f} / {row.direct_odds_ratio_span:.2f} '
      f'/ {row.via_exposure_odds_ratio_span:.2f} |'
      for row in khb_decomposition.itertuples()],
    '',
    'Log-odds coefficients with cluster-robust standard errors. The total is '
    'the characteristic\'s coefficient when log F* is replaced by its residual '
    'from a linear regression on all other regressors, which puts it on the '
    'scale of column (1); the direct component is its column (1) coefficient; '
    'the component via F* is their difference, equal to the log F* '
    'coefficient times the characteristic\'s slope on log F*, with a '
    'delta-method standard error. The share via F* exceeds 100% when the '
    'direct component has the opposite sign. Odds ratios are over the Table 1 '
    'spans: tile versus asphalt, one decade, or the interquartile range of the '
    'logged predictors. The decomposition is descriptive: F* is built from '
    'the outcomes of neighboring structures, so the direct component is not a '
    'causal direct effect. \\*\\*\\**P* < 0.001, \\*\\**P* < 0.01, '
    '\\**P* < 0.05.',
]
khb_markdown = '\n'.join(khb_markdown_lines)
(RESULTS / 'misc_khb_exposure_decomposition.md').write_text(
    khb_markdown + '\n', encoding='utf-8')
display(Markdown(khb_markdown))
print('saved', RESULTS / 'misc_khb_exposure_decomposition.csv')
print('saved', RESULTS / 'misc_khb_exposure_decomposition.md')

### Do attributes offset more exposure once a structure is damaged?

Table 1 suggests that vegetation and vintage are worth more exposure at the
escalation stage than at damage onset, and roofing less. The exposure-tolerance
ratio is a ratio of coefficients from one model, so the logit rescaling between
stages cancels and the two stages can be compared directly; but the two
estimates are correlated, because the escalation sample is the damaged subset
of the same structures. This cell bootstraps the difference: 250 m grid cells
are resampled within each fire, both stage models are refitted on every draw,
and the escalation tolerance is divided by the any-damage tolerance. Spans are
fixed at the Table 1 values, and the number of draws follows `N_BOOT`.

In [ ]:
import statsmodels.formula.api as smf

# Do the attributes offset more exposure once a structure is damaged? The
# exposure-tolerance ratio exp(-gamma * span / beta) is a ratio of
# coefficients from one model, so the logit rescaling between stages cancels
# and the any-damage and escalation ratios can be compared directly. Their
# estimates are correlated, because the escalation sample is the damaged
# subset of the same structures, so the difference is bootstrapped: 250 m grid
# cells are resampled within each fire and both models are refitted on every
# draw. Spans are fixed at the Table 1 values.
STAGE_TERMS = [
    (TILE_TERM, 'Tile roof, versus asphalt', 1.),
    ('log_ndvi', 'Near-structure NDVI, interquartile increase', None),
    ('year_decade', 'Building vintage, per decade newer', 1.),
    ('log_influence_depth', 'Distance into built environment, interquartile',
     None),
    ('log_n_visible', 'Connected neighbors, interquartile', None),
    ('log_footprint', 'Footprint area, interquartile', None),
]
stage_spans = {
    term: span if span is not None else float(
        np.subtract(*np.percentile(regression_sample[term], [75, 25])))
    for term, _, span in STAGE_TERMS}
STAGE_FORMULA = f'{BASE_TERMS} + {DEPTH_TERM}'


def _log_tolerances(damage_model, escalation_model) -> dict[str, float]:
    """Log exposure-tolerance ratio of each term at each stage."""
    values = {}
    for term, _, _ in STAGE_TERMS:
        for stage, model in [('damage', damage_model),
                             ('escalation', escalation_model)]:
            values[f'{term}|{stage}'] = float(
                -model.params[term] * stage_spans[term]
                / model.params['log_F_star'])
    return values


fitted_by_key = dict((key, model) for key, _, _, model in FITTED_MODELS)
stage_point = _log_tolerances(fitted_by_key['any_damage'],
                              fitted_by_key['escalation'])

stage_rng = np.random.default_rng(BOOTSTRAP_SEED)
stage_clusters = {fire_name: group.grid_id.unique()
                  for fire_name, group in regression_sample.groupby('fire')}
stage_cluster_rows = regression_sample.groupby('grid_id').indices
stage_draws, stage_failures = [], 0
for draw in range(N_BOOT):
    chosen = np.concatenate([
        stage_rng.choice(clusters, size=len(clusters), replace=True)
        for clusters in stage_clusters.values()])
    boot = regression_sample.iloc[
        np.concatenate([stage_cluster_rows[cluster] for cluster in chosen])]
    try:
        damage_fit = smf.logit(f'any_damage ~ {STAGE_FORMULA}',
                               data=boot).fit(disp=0, maxiter=200)
        escalation_fit = smf.logit(
            f'is_destroyed ~ {STAGE_FORMULA}',
            data=boot[boot.any_damage.eq(1)]).fit(disp=0, maxiter=200)
    except Exception:  # a draw without a rare roof class can fail to fit
        stage_failures += 1
        continue
    stage_draws.append({'draw': draw,
                        **_log_tolerances(damage_fit, escalation_fit)})
stage_draws = pd.DataFrame(stage_draws)

stage_rows = []
for term, label, _ in STAGE_TERMS:
    damage = stage_draws[f'{term}|damage']
    escalation = stage_draws[f'{term}|escalation']
    difference = escalation - damage
    point_damage = stage_point[f'{term}|damage']
    point_escalation = stage_point[f'{term}|escalation']
    stage_rows.append({
        'term': term, 'term_label': label,
        'tolerance_damage': np.exp(point_damage),
        'tolerance_damage_lo': np.exp(damage.quantile(.025)),
        'tolerance_damage_hi': np.exp(damage.quantile(.975)),
        'tolerance_escalation': np.exp(point_escalation),
        'tolerance_escalation_lo': np.exp(escalation.quantile(.025)),
        'tolerance_escalation_hi': np.exp(escalation.quantile(.975)),
        'log_difference': point_escalation - point_damage,
        'ratio_escalation_to_damage': np.exp(point_escalation - point_damage),
        'ratio_lo': np.exp(difference.quantile(.025)),
        'ratio_hi': np.exp(difference.quantile(.975)),
        # Two-sided bootstrap P for no difference, floored at one draw.
        'p_value': max(2 * min((difference <= 0).mean(),
                               (difference >= 0).mean()),
                       1 / len(difference)),
        'draws': len(difference),
    })
stage_tolerance_test = pd.DataFrame(stage_rows)
stage_tolerance_test.to_csv(
    RESULTS / 'misc_stage_tolerance_bootstrap.csv', index=False)
stage_draws.to_csv(
    RESULTS / 'misc_stage_tolerance_bootstrap_draws.csv', index=False)

def _bootstrap_p(p_value: float, draws: int) -> str:
    """P at the floor of one draw means no draw crossed zero."""
    return f'≤ {1 / draws:.3f}' if p_value <= 1 / draws else f'{p_value:.3f}'


stage_markdown_lines = [
    '**Exposure-tolerance ratios by stage, with a cluster-bootstrap test of '
    'the difference.**',
    '',
    '| Attribute | Any damage | Destruction given damage | Ratio, '
    'escalation / damage | Bootstrap *P* |',
    '|---|:--:|:--:|:--:|:--:|',
    *[f'| {row.term_label} '
      f'| {row.tolerance_damage:.2f} ({row.tolerance_damage_lo:.2f}–'
      f'{row.tolerance_damage_hi:.2f}) '
      f'| {row.tolerance_escalation:.2f} ({row.tolerance_escalation_lo:.2f}–'
      f'{row.tolerance_escalation_hi:.2f}) '
      f'| {row.ratio_escalation_to_damage:.2f} ({row.ratio_lo:.2f}–'
      f'{row.ratio_hi:.2f}) '
      f'| {_bootstrap_p(row.p_value, row.draws)} |'
      for row in stage_tolerance_test.itertuples()],
    '',
    f'Exposure-tolerance ratios from models (2) and (3), over the Table 1 '
    f'spans, with 95% percentile intervals from {len(stage_draws)} '
    f'fire-stratified bootstrap draws of 250 m grid cells '
    f'({stage_failures} draws failed to fit and were dropped). The ratio '
    'column divides the escalation tolerance by the any-damage tolerance: '
    'above one, the attribute is associated with greater exposure tolerance '
    'once a structure is damaged; below one, with less. Vegetation is '
    'reported as an increase, so its ratio below one means a reduction is '
    'worth more exposure at the second stage. Escalation is estimated on '
    'damaged structures and is descriptive.',
]
stage_markdown = '\n'.join(stage_markdown_lines)
(RESULTS / 'misc_stage_tolerance_bootstrap.md').write_text(
    stage_markdown + '\n', encoding='utf-8')
display(Markdown(stage_markdown))
print('saved', RESULTS / 'misc_stage_tolerance_bootstrap.csv')